In [6]:
# ============================================================
# Ghosh (supply-driven) shocks to Copper (by country, one at a time)
# -> Measure impact on FR wind and FR solar PV outputs
# ============================================================
import os, re
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymrio

import matplotlib as mpl

from datetime import datetime

import glob
import numpy as np

import textwrap

import time

from pathlib import Path

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, get_cmap

import zipfile

from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib.colors import LinearSegmentedColormap

import seaborn as sns

In [10]:
# -----------------------------
# 0) Configure EXIOBASE path
# -----------------------------
path_zip = r"C:\Users\Dyde-nairn\Documents\python\_input_output_dependencies\IOT_2022_pxp.v3.10.1.zip"

if not os.path.exists(path_zip):
    raise FileNotFoundError(
        f"EXIOBASE file not found at:\n{path_zip}\n"
        "Please update 'path_zip' to the location of your EXIOBASE3 PXP .zip file."
    )
# -----------------------------
# 1) Load data
# -----------------------------
exio3 = pymrio.parse_exiobase3(path=path_zip)  # provides A, x, Y, etc.
exio3.calc_all()

In [11]:
# -*- coding: utf-8 -*-
"""
ALL-country Ghosh shocks -> ABSOLUTE changes in Solar PV output (Million EUR)
Robust, self-contained script with:
  • Auto-detection of MultiIndex order (Country, Industry vs. Industry, Country)
  • Discovery & fuzzy mapping of upstream SHOCK_INDUSTRIES to actual labels
  • Mask-based resolution of Solar PV labels per destination country
  • 1% supply cuts ONE COUNTRY AT A TIME per shock industry (batched solve)
  • Progress logging, diagnostics, and CSV outputs:
      - ghosh_shocks_to_solar/master__solar_impacts_allcountries__<pct>__<timestamp>.csv
      - ghosh_shocks_to_solar/per_country/<DEST>__solar_impacts_allcountries__<pct>__<timestamp>.csv

Prereqs already loaded in memory:
    exio3.A : pd.DataFrame (MultiIndex rows/cols)
    exio3.x : pd.Series or pd.DataFrame (MultiIndex index)
"""
# ------------------ Config ------------------
shock_pct = 0.01  # 1% supply cut, applied ONE COUNTRY AT A TIME
out_dir_base = "ghosh_shocks_to_solar"
per_country_dir = os.path.join(out_dir_base, "per_country")
os.makedirs(per_country_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")

# Progress controls
VERBOSE_PER_COUNTRY = True   # print status per country
PROGRESS_EVERY = 5           # print every N-th country to reduce noise
SHOW_INDUSTRY_TIMING = True  # print timing per shock industry

# Optional toggles
EXCLUDE_ZERO_SHOCKERS = False         # skip countries with zero output in shocked industry
SKIP_EMPTY_DEST_FILES = False         # skip writing per-country CSVs that are all-NaN
PREFER_PHOTOVOLTAIC_ONLY = True       # prefer PV-only; if no PV coverage, fallback to broader solar-electric

# ------------------ Helpers ------------------
def safe_slug(s):
    return "".join(ch if ch.isalnum() or ch in ("_", "-", ".") else "_" for ch in str(s))

def _norm(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s)).strip().lower()
    s = s.replace("aluminium", "aluminum")
    s = s.replace("alumina", "aluminum")
    s = s.replace("bauxite", "aluminum")
    s = s.replace("photovoltaics", "photovoltaic")
    s = s.replace("pv", "photovoltaic")
    s = s.replace("ores", "ore").replace("concentrates", "concentrate")
    s = s.replace("nec", "n.e.c").replace("n.e.c.", "n.e.c")
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s\.\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _contains_all(s: str, tokens):
    s = _norm(s); return all(t in s for t in tokens)

def _contains_any(s: str, tokens):
    s = _norm(s); return any(t in s for t in tokens)

def best_match(universe, token_groups_and_first=True, *token_groups):
    """
    Choose the best matching label from 'universe' (iterable of strings) given multiple token groups.
    Strategy:
      - AND matching first for each group; fallback to OR matching using union of tokens.
      - Score = (#tokens matched) - small penalty for length difference to prefer tighter matches.
    Returns (label or None, score).
    """
    def score_label(label, tokens):
        s = _norm(label)
        matched = sum(1 for t in tokens if t in s)
        penalty = len(s) / 1000.0
        return matched - penalty

    if token_groups_and_first:
        candidates = []
        for tg in token_groups:
            hits = [l for l in universe if _contains_all(l, tg)]
            candidates.extend((h, score_label(h, tg)) for h in hits)
        if candidates:
            return sorted(candidates, key=lambda x: x[1], reverse=True)[0]

    union_tokens = sorted(set(t for g in token_groups for t in g))
    candidates = [(l, score_label(l, union_tokens)) for l in universe if _contains_any(l, union_tokens)]
    if candidates:
        return sorted(candidates, key=lambda x: x[1], reverse=True)[0]
    return (None, -1e9)

# ------------------ Core matrices ------------------
# Technical coefficients (MultiIndex rows/cols)
A_df = exio3.A.copy()

# Baseline outputs x (Series with MultiIndex). Handle Series/DataFrame variants.
if isinstance(exio3.x, pd.Series):
    x_base = exio3.x.copy().reindex(A_df.index)
else:
    # If exio3.x is a DataFrame (layers/valuations), sum across columns as robust fallback
    tmp = exio3.x.sum(axis=1)
    tmp.index = exio3.x.index
    x_base = tmp.reindex(A_df.index)

# Build Z from A and x: z_ij = A_ij * x_j
x_cols = x_base.reindex(A_df.columns)                 # align outputs with columns
Z_df = A_df.mul(x_cols, axis=1)

# B = row-normalized Z: B_ij = z_ij / x_i
x_rows = x_base.reindex(Z_df.index).replace(0, np.nan)  # avoid div-by-zero
B_df = Z_df.div(x_rows, axis=0).fillna(0.0)

# Identity and (I - B)
I_df = pd.DataFrame(np.eye(len(B_df)), index=B_df.index, columns=B_df.columns)
index_MI = B_df.index
M = (I_df - B_df).values

# Baseline primary inputs/value added s
s_base = (I_df - B_df).dot(x_base)

# Diagnostics
N = len(B_df)
approx_gb = (N * N * 8) / (1024**3)
rho_B = float(np.max(np.abs(np.linalg.eigvals(B_df.values))))
resid = (I_df - B_df).dot(x_base.reindex(index_MI)) - s_base.reindex(index_MI)
print(f"[Info] N={N}, dense float64 ≈ {approx_gb:.2f} GB per matrix")
print(f"[Diagnostics] Spectral radius(B) = {rho_B:.6f}")
print(f"[Diagnostics] ||(I-B)x - s||_∞ = {float(np.abs(resid).max()):.3e}")

# ------------------ Detect MultiIndex order ------------------
countries_lvl0 = pd.Index(index_MI.get_level_values(0)).unique()
industries_lvl1 = pd.Index(index_MI.get_level_values(1)).unique()

def detect_order(index_MI, countries, industries):
    for cc in list(countries)[:min(10, len(countries))]:
        for ind in list(industries)[:min(20, len(industries))]:
            if (cc, ind) in index_MI:
                return "CI"  # Country, Industry
            if (ind, cc) in index_MI:
                return "IC"  # Industry, Country
    return "CI"  # fallback

ORDER = detect_order(index_MI, countries_lvl0, industries_lvl1)
print(f"[Detect] MultiIndex order: {'(Country, Industry)' if ORDER=='CI' else '(Industry, Country)'}")

def mk_label(country, industry):
    return (country, industry) if ORDER == "CI" else (industry, country)

# Define convenience handles for levels
level_country = 0 if ORDER == "CI" else 1
level_industry = 1 if ORDER == "CI" else 0

# Destinations (countries) & industry universe
dest_countries = list(pd.Index(index_MI.get_level_values(level_country)).unique())
industries_universe = pd.Index(index_MI.get_level_values(level_industry)).unique()
print(f"[Discover] Found {len(industries_universe)} unique industry names in index_MI.")

# ------------------ Show solar-like labels ------------------
solar_like_examples = [s for s in industries_universe if _contains_any(s, ("solar", "photovoltaic"))]
print(f"[Discover] Solar-like industry examples (up to 15):")
for s in solar_like_examples[:15]:
    print("  •", s)

# ------------------ Robust shock industry mapping ------------------
shock_token_groups = {
    # Primary (ores)
    "Aluminium Ore": [
        ("aluminum", "ore"), ("bauxite", "ore"), ("aluminum", "concentrate")
    ],
    "Copper Ore": [
        ("copper", "ore"), ("copper", "concentrate")
    ],
    "Lead, Zinc & Tin Ore": [
        ("lead", "ore"), ("zinc", "ore"), ("tin", "ore"), ("lead", "zinc", "tin", "ore")
    ],
    "Nickel Ore": [
        ("nickel", "ore"), ("nickel", "concentrate")
    ],
    "Other Non-Ferrous Metal Ore": [
        ("non", "ferrous", "metal", "ore"), ("other", "non", "ferrous", "ore")
    ],
    "Precious Metal Ore": [
        ("precious", "metal", "ore"), ("gold", "ore"), ("silver", "ore")
    ],
    # Secondary (metal products)
    "Aluminium Products": [
        ("aluminum", "product"), ("aluminum",), ("basic", "aluminum")
    ],
    "Copper Products": [
        ("copper", "product"), ("copper",)
    ],
    "Basic Iron, Steel & Ferro-Alloy Products": [
        ("iron", "steel", "ferro"), ("basic", "iron", "steel"), ("steel", "ferro", "alloy")
    ],
    "Lead, Zinc & Tin Products": [
        ("lead", "zinc", "tin", "product"), ("lead", "product"), ("zinc", "product"), ("tin", "product")
    ],
    "Other Non-Ferrous Mineral Products": [
        ("non", "ferrous", "product"), ("other", "non", "ferrous", "product")
    ],
    "Precious Metal Products": [
        ("precious", "metal", "product"), ("precious", "metals")
    ],
    # Tertiary (machinery/fabricated metal)
    "Electrical Machinery and Apparatus": [
        ("electrical", "machinery"), ("electrical", "apparatus")
    ],
    "Fabricated Metal Products": [
        ("fabricated", "metal", "product"), ("fabricated", "metal")
    ],
    "Machinery and Equipment": [
        ("machinery", "equipment"), ("machinery",)
    ],
}

resolved_shocks = []
not_resolved = []
for display_name, token_groups in shock_token_groups.items():
    label, score = best_match(industries_universe, True, *token_groups)
    if label is None:
        print(f"[Map] ⚠️ No match for '{display_name}'")
        not_resolved.append(display_name)
    else:
        print(f"[Map] {display_name} -> '{label}' (score={score:.2f})")
        resolved_shocks.append(label)

_seen = set()
SHOCK_INDUSTRIES = [x for x in resolved_shocks if not (x in _seen or _seen.add(x))]
print(f"[Shock list] Resolved {len(SHOCK_INDUSTRIES)} upstream labels.")
if not_resolved:
    print("[Warn] Unresolved categories:", not_resolved)

# ------------------ Resolve Solar PV label (mask-based, robust) ------------------
def find_best_solar_label(industries_universe, prefer_pv=True):
    cand_all = [
        ind for ind in industries_universe
        if (("solar" in _norm(ind) or "photovoltaic" in _norm(ind))
            and ("electric" in _norm(ind) or "electricity" in _norm(ind)))
    ]
    if prefer_pv:
        pv_cands = [ind for ind in cand_all if "photovoltaic" in _norm(ind)]
        cands_to_try = pv_cands if pv_cands else cand_all
    else:
        cands_to_try = cand_all
    # Compute coverage (how many countries actually have (country, ind) rows)
    best = None
    best_cover = -1
    coverage = {}
    for ind in cands_to_try:
        mask_ind = (index_MI.get_level_values(level_industry) == ind)
        countries_with_ind = pd.Index(index_MI.get_level_values(level_country)[mask_ind]).unique()
        coverage[ind] = (len(countries_with_ind), countries_with_ind[:10].tolist())
        if len(countries_with_ind) > best_cover:
            best_cover = len(countries_with_ind)
            best = ind
    return best, coverage

best_solar, coverage_map = find_best_solar_label(industries_universe, prefer_pv=PREFER_PHOTOVOLTAIC_ONLY)
print("\n[Coverage] Solar label coverage across countries:")
for ind, (cover, sample_cc) in coverage_map.items():
    print(f"  - '{ind}': {cover} countries (sample: {sample_cc})")

if best_solar is None or coverage_map.get(best_solar, (0, []))[0] == 0:
    raise RuntimeError(
        "No Solar PV/solar-electric label present as (country, industry) in index_MI. "
        "Inspect coverage above and adjust matching tokens or dataset."
    )

print(f"\n[Use] Solar industry selected -> '{best_solar}' "
      f"with coverage {coverage_map[best_solar][0]} countries")

# Per-country Solar PV labels using masks
target_labels = {}
for cc in dest_countries:
    mask_cc = (index_MI.get_level_values(level_country) == cc)
    mask_ind = (index_MI.get_level_values(level_industry) == best_solar)
    mask = mask_cc & mask_ind
    pos = np.flatnonzero(mask)
    target_labels[cc] = index_MI[pos[0]] if pos.size > 0 else None

n_solar_ok = sum(1 for cc in dest_countries if target_labels[cc] is not None)
print(f"[Match] Solar labels found for {n_solar_ok}/{len(dest_countries)} countries.")

# Row positions for extraction later
pos_map = {lab: i for i, lab in enumerate(index_MI)}
dest_pos = {
    cc: (pos_map[target_labels[cc]] if target_labels[cc] is not None else None)
    for cc in dest_countries
}

# ------------------ Shocks & batch solve with progress ------------------
records_all = []
idx = pd.IndexSlice

t_build_start = time.time()
for si_idx, shock_industry in enumerate(SHOCK_INDUSTRIES, start=1):
    t_industry_start = time.time()
    print(f"\n[Run] {si_idx}/{len(SHOCK_INDUSTRIES)} Shock industry -> {shock_industry}", flush=True)

    # Baseline output by country of shocked industry (for reporting only)
    try:
        exio_x = exio3.x
        if isinstance(exio_x, pd.Series):
            shock_out_series = exio_x.loc[idx[:, shock_industry]].droplevel(1)
        else:
            tmp = exio_x.loc[idx[:, shock_industry], :]
            shock_out_series = tmp.sum(axis=1).droplevel(1)
    except Exception:
        # Fallback: use x_base
        sub = x_base.loc[idx[:, shock_industry]]
        shock_out_series = sub.droplevel(1)

    # Ensure all destination countries present
    shock_out_series = shock_out_series.reindex(dest_countries).fillna(0.0)

    # Shock labels: ALL countries (order-aware)
    shock_labels = [mk_label(cc, shock_industry) for cc in dest_countries]
    if EXCLUDE_ZERO_SHOCKERS:
        shock_labels = [mk_label(cc, shock_industry) for cc in dest_countries if shock_out_series.loc[cc] != 0.0]

    # Keep only those present in system index
    existing_labels = [lab for lab in shock_labels if lab in pos_map]
    if not existing_labels:
        print(f"[Warn] No labels in system index for '{shock_industry}'. Skipping.")
        continue
    existing_pos = [pos_map[lab] for lab in existing_labels]

    # Baselines aligned to matrix order
    s0 = s_base.reindex(index_MI)

    # Batched RHS ΔS (n x k) with sign-aware magnitude cut
    n, k = len(index_MI), len(existing_labels)
    dS = np.zeros((n, k), dtype=float)
    for j, irow in enumerate(existing_pos):
        base_val = float(s0.iloc[irow])
        dS[irow, j] = -shock_pct * abs(base_val)

    # Solve (I - B) ΔX = ΔS (no explicit inverse; lower RAM)
    t_solve_start = time.time()
    dX = np.linalg.solve(M, dS)  # n x k
    t_solve_end = time.time()
    print(f"[Timing] Solve for {shock_industry}: {t_solve_end - t_solve_start:.2f}s (k={k} shock countries)", flush=True)

    # Column map: shock label -> column index j in dX
    col_index_map = {lab: j for j, lab in enumerate(existing_labels)}

    # Build records; print country progress
    for cc_idx, cc in enumerate(dest_countries, start=1):
        rec = {
            "Shock_Industry": shock_industry,
            "Shock_Country": cc,
            "Shock_Output_EUR": float(shock_out_series.loc[cc]),
        }

        lab = mk_label(cc, shock_industry)
        if lab not in col_index_map:
            for dest in dest_countries:
                rec[f"abs_{dest}_EUR"] = np.nan
            records_all.append(rec)
            if VERBOSE_PER_COUNTRY and (cc_idx % PROGRESS_EVERY == 0):
                print(f"[{shock_industry}] {cc}: label missing — skipped ({cc_idx}/{len(dest_countries)})", flush=True)
            continue

        jcol = col_index_map[lab]
        for dest in dest_countries:
            rpos = dest_pos.get(dest, None)
            rec[f"abs_{dest}_EUR"] = float(dX[rpos, jcol]) if rpos is not None else np.nan

        records_all.append(rec)
        if VERBOSE_PER_COUNTRY and (cc_idx % PROGRESS_EVERY == 0):
            print(f"[{shock_industry}] {cc} done ({cc_idx}/{len(dest_countries)})", flush=True)

    if SHOW_INDUSTRY_TIMING:
        print(f"[Timing] {shock_industry} finished in {time.time() - t_industry_start:.2f}s", flush=True)

print(f"\n[Total timing] Shock pass finished in {time.time() - t_build_start:.2f}s", flush=True)

# ------------------ Save master & per-country CSVs ------------------
df_master = pd.DataFrame(records_all)

if df_master.empty:
    print("[Error] No records produced. Check mappings and detected order.")
    # Save diagnostics under both orders to help debugging
    diag_rows = []
    for shock_industry in SHOCK_INDUSTRIES:
        for cc in dest_countries:
            diag_rows.append({
                "Shock_Industry": shock_industry,
                "Shock_Country": cc,
                "Has_Label_CI": ((cc, shock_industry) in index_MI),
                "Has_Label_IC": ((shock_industry, cc) in index_MI),
                "Order_Used": ORDER,
                "Has_Solar_Label": (target_labels.get(cc) is not None),
            })
    df_diag = pd.DataFrame(diag_rows)
    diag_path = os.path.join(out_dir_base, f"diagnostics__{ts}.csv")
    df_diag.to_csv(diag_path, index=False, encoding="utf-8")
    print(f"[Saved diagnostics] {diag_path}")
else:
    df_master = df_master.sort_values(["Shock_Industry", "Shock_Country"], ascending=True)
    master_name = f"master__solar_impacts_allcountries__{int(shock_pct*100)}pct__{ts}.csv"
    master_path = os.path.join(out_dir_base, master_name)
    df_master.to_csv(master_path, index=False, encoding="utf-8")
    print(f"[Saved master] {master_path}")

    # Per-country CSVs (Solar-only)
    for dest in dest_countries:
        colname = f"abs_{dest}_EUR"
        if colname not in df_master.columns:
            print(f"[Skip per-country] Missing column for destination={dest}")
            continue

        df_dest = (
            df_master[["Shock_Industry", "Shock_Country", "Shock_Output_EUR", colname]]
            .rename(columns={colname: "Abs_Change_EUR"})
            .sort_values(["Shock_Industry", "Shock_Country"], ascending=True)
        )

        if SKIP_EMPTY_DEST_FILES and not np.isfinite(df_dest["Abs_Change_EUR"]).any():
            print(f"[Skip per-country] All-NaN impacts for destination={dest}")
            continue

        fname = f"{safe_slug(dest)}__solar_impacts_allcountries__{int(shock_pct*100)}pct__{ts}.csv"
        fpath = os.path.join(per_country_dir, fname)
        df_dest.to_csv(fpath, index=False, encoding="utf-8")
        print(f"[Saved per-country] {fpath}")

[Info] N=9800, dense float64 ≈ 0.72 GB per matrix
[Diagnostics] Spectral radius(B) = 0.882143
[Diagnostics] ||(I-B)x - s||_∞ = 0.000e+00
[Detect] MultiIndex order: (Country, Industry)
[Discover] Found 200 unique industry names in index_MI.
[Discover] Solar-like industry examples (up to 15):
  • Electricity by solar photovoltaic
  • Electricity by solar thermal
[Map] Aluminium Ore -> 'Aluminium ores and concentrates' (score=1.97)
[Map] Copper Ore -> 'Copper ores and concentrates' (score=1.97)
[Map] Lead, Zinc & Tin Ore -> 'Lead, zinc and tin ores and concentrates' (score=3.96)
[Map] Nickel Ore -> 'Nickel ores and concentrates' (score=1.97)
[Map] Other Non-Ferrous Metal Ore -> 'Other non-ferrous metal ores and concentrates' (score=3.96)
[Map] Precious Metal Ore -> 'Precious metal ores and concentrates' (score=2.97)
[Map] Aluminium Products -> 'Aluminium and aluminium products' (score=1.97)
[Map] Copper Products -> 'Copper products' (score=1.99)
[Map] Basic Iron, Steel & Ferro-Alloy Produ

In [12]:

# -*- coding: utf-8 -*-
"""
Create per-destination heatmaps from the 'master' CSV produced by the Ghosh shock sweep.

Master CSV columns expected:
  - 'Shock_Industry', 'Shock_Country', 'Shock_Output_EUR'
  - one column per destination: 'abs_<CC>_EUR' (absolute change in Solar PV for <CC>, Million EUR)

Output folder structure:
  heatmaps_from_master/
    <DEST>/
      heatmap_abs_changes_negonly_white0_<DEST>.png
      matrix_<DEST>.csv
"""
# ==================== CONFIG ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv"  # <-- set actual path
OUT_DIR_BASE = "heatmaps_from_master"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

# If None → all destinations found in master; or pass a list like ["CN","FR","DE","JP","US"]
DEST_LIST = None

# Rows in the heatmap (choose ~15–30 for readability). If None -> keep all.
TOP_COUNTRIES_FOR_HEATMAP = 49

FIGSIZE = (18, 10)   # per-destination heatmap size
NEGATIVE_ONLY = True # white=0, darker blue for more negative; positives clipped to 0
ANNOTATE = True
ANNOTATION_FMT = "{:.3f}"

# --- Column groups (similar to your example: tertiary -> secondary -> primary) ---
PRIORITY_FIRST = [
    "Electrical machinery and apparatus n.e.c. (31)",
    "Fabricated metal products, except machinery and equipment (28)",
    "Machinery and equipment n.e.c. (29)",
]
METAL_PRODUCTS_BLOCK = [
    "Aluminium and aluminium products",
    "Basic iron and steel and of ferro-alloys and first products thereof",
    "Copper products",
    "Lead, zinc and tin and products thereof",
    "Other non-ferrous metal products",
    # If you want to include the ores here, move them to the 'rest' set or add another block
    # "Precious metal ores and concentrates",  # typically primary, so keep in 'rest'
]

# --- Continent mapping (extend to your code set; supports 2- or 3-letter codes) ---
CONTINENT_MAP = {
    # Europe (2-letter)
    "AT":"Europe","BE":"Europe","BG":"Europe","CH":"Europe","CZ":"Europe","DE":"Europe","DK":"Europe",
    "ES":"Europe","FI":"Europe","FR":"Europe","GB":"Europe","GR":"Europe","HR":"Europe","HU":"Europe",
    "IE":"Europe","IT":"Europe","NL":"Europe","NO":"Europe","PL":"Europe","PT":"Europe","RO":"Europe",
    "RU":"Europe","SE":"Europe","SI":"Europe","SK":"Europe","WE":"Europe",
    # Americas (2-letter)
    "BR":"Americas","CA":"Americas","MX":"Americas","US":"Americas","WL":"Americas",
    # Asia (2-letter)
    "CN":"Asia","ID":"Asia","IN":"Asia","JP":"Asia","KR":"Asia","TW":"Asia","TR":"Asia","WA":"Asia",
    # Oceania
    "AU":"Oceania",
    # Africa
    "ZA":"Africa","WF":"Africa",
    # Middle East
    "WM":"Middle East",

    # Examples of 3-letter codes (extend if your master uses 3-letter):
    "FRA":"Europe","DEU":"Europe","ESP":"Europe","ITA":"Europe","GBR":"Europe","NLD":"Europe","SWE":"Europe",
    "CHE":"Europe","NOR":"Europe","POL":"Europe","PRT":"Europe","AUT":"Europe","BEL":"Europe","CZE":"Europe",
    "HUN":"Europe","SVK":"Europe","SVN":"Europe","IRL":"Europe","ROU":"Europe","BGR":"Europe","DNK":"Europe",
    "FIN":"Europe","LUX":"Europe","GRC":"Europe",
    "USA":"Americas","CAN":"Americas","MEX":"Americas","BRA":"Americas","CHL":"Americas","ARG":"Americas",
    "CHN":"Asia","JPN":"Asia","KOR":"Asia","IND":"Asia","IDN":"Asia","THA":"Asia","VNM":"Asia","TUR":"Asia",
    "AUS":"Oceania","NZL":"Oceania",
    "ZAF":"Africa","MAR":"Africa","EGY":"Africa",
}
CONTINENT_ORDER = ["Europe", "Americas", "Asia", "Middle East", "Africa", "Oceania"]

# ==================== HELPERS ====================
def _norm(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s)).strip().lower()
    s = s.replace("&", " and ")
    s = s.replace("aluminium", "aluminum")
    s = s.replace("photovoltaics", "photovoltaic")
    return s

def classify_industry(ind_name: str):
    """
    Return one of {'tertiary','secondary','primary'} via simple token search.
    This is best-effort; we enforce ordering via explicit lists below.
    """
    s = _norm(ind_name)
    if any(k in s for k in ["machinery","apparatus","equipment","n.e.c"]):
        return "tertiary"
    if any(k in s for k in ["product","basic","precious","non-ferrous product"]):
        return "secondary"
    if any(k in s for k in ["ore","concentrate","mining"]):
        return "primary"
    # fallback
    return "tertiary"

def compute_ordered_columns_and_blocks(present_cols):
    """
    Return ordered_cols and column group blocks aligned to present_cols:
      ordered_cols: [priority_first (present)] + [metal_products (present)] + [rest alphabetical]
      col_blocks: list of (group_name, start_idx, end_idx)
    """
    present_set = set(present_cols)
    priority = [c for c in PRIORITY_FIRST if c in present_set]
    metals   = [c for c in METAL_PRODUCTS_BLOCK if c in present_set]
    rest     = sorted([c for c in present_cols if c not in set(PRIORITY_FIRST + METAL_PRODUCTS_BLOCK)])

    ordered_cols = priority + metals + rest

    # blocks for banding/labels
    blocks = []
    start = 0
    if len(priority) > 0:
        blocks.append(("Tertiary", start, start + len(priority) - 1))
        start += len(priority)
    if len(metals) > 0:
        blocks.append(("Secondary", start, start + len(metals) - 1))
        start += len(metals)
    if len(rest) > 0:
        # Heuristic: assume remaining 'rest' are primary (mostly ores & concentrates)
        blocks.append(("Primary", start, start + len(rest) - 1))

    return ordered_cols, blocks

def infer_destination_list(master_columns):
    """Return list of destination country codes from 'abs_<CC>_EUR' columns."""
    dests = []
    for c in master_columns:
        if isinstance(c, str) and c.startswith("abs_") and c.endswith("_EUR"):
            dests.append(c[len("abs_"):-len("_EUR")])
    return dests

def order_rows_by_continent(rows, within_continent="alpha"):
    """
    Reorder the given 'rows' (country codes) by CONTINENT_ORDER.
    within_continent: 'alpha' or 'keep'
    """
    buckets = {cont: [] for cont in CONTINENT_ORDER}
    others = []
    for r in rows:
        cont = CONTINENT_MAP.get(str(r))
        if cont in buckets:
            buckets[cont].append(r)
        else:
            others.append(r)
    if within_continent == "alpha":
        for cont in buckets:
            buckets[cont].sort()
    ordered = []
    for cont in CONTINENT_ORDER:
        ordered.extend(buckets[cont])
    ordered.extend(others)
    return ordered

def _continent_blocks_for_rows(rows):
    """
    Given an ordered list of rows (country codes), return
    (continent_name, start_idx, end_idx) for contiguous blocks.
    """
    blocks = []
    i = 0
    n = len(rows)
    while i < n:
        cont = CONTINENT_MAP.get(str(rows[i]), "Other")
        start = i
        i += 1
        while i < n and CONTINENT_MAP.get(str(rows[i]), "Other") == cont:
            i += 1
        end = i - 1
        blocks.append((cont, start, end))
    return blocks

def _draw_column_group_decorations(ax, mat2, col_blocks):
    """Add vertical banding, separators, and top labels for column groups."""
    # light banding
    for idx, (gname, start, end) in enumerate(col_blocks):
        if idx % 2 == 0:
            ax.axvspan(start - 0.5, end + 0.5, facecolor="lightgrey", alpha=0.08, zorder=0)
    # separators
    for k, (_, start, end) in enumerate(col_blocks):
        if k < len(col_blocks) - 1:
            ax.axvline(end + 0.5, color="black", lw=1.2, alpha=0.35)
    # top labels
    ymax = -0.5
    for gname, start, end in col_blocks:
        mid = (start + end) / 2.0
        ax.text(
            mid, ymax - 0.15, gname,
            va="top", ha="center", fontsize=10, color="dimgray", rotation=0, alpha=0.95, clip_on=False,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.6),
            transform=ax.transData
        )
    plt.subplots_adjust(top=0.88)

def build_matrix_from_master(master_df, dest_country, top_n_rows=None):
    """
    From the master CSV, build the matrix:
      rows = Shock_Country
      cols = Shock_Industry
      values = abs change for dest_country (Million EUR)
    Apply Top-N filter by aggregated |impact| across industries if requested.
    """
    val_col = f"abs_{dest_country}_EUR"
    if val_col not in master_df.columns:
        raise KeyError(f"Destination column '{val_col}' not found in master CSV.")

    df = master_df[["Shock_Country", "Shock_Industry", val_col]].copy()
    df.rename(columns={val_col: "value"}, inplace=True)

    # Pivot
    mat = df.pivot(index="Shock_Country", columns="Shock_Industry", values="value")

    # Top-N rows by aggregate magnitude
    if top_n_rows is not None and top_n_rows > 0:
        mag = mat.abs().sum(axis=1).sort_values(ascending=False)
        keep_rows = list(mag.head(top_n_rows).index)
        mat = mat.loc[keep_rows, :]

    return mat



def plot_heatmap_negative_only(matrix, title, outpath, figsize=(18, 10), annotate=True, annotation_fmt="{:.1f}"):
    """
    Negative-only heatmap with continent grouping (rows) and 3-block column grouping:
      - vmin from robust percentile of negatives
      - vmax = 0 → 0 is white (Blues_r), positives clipped to 0
      - annotate cells
      - continent banding, right labels
      - column group banding, separators, top labels
    """
    mat = matrix.copy()

    # Clean tiny -0.0 noise -> exactly 0
    # (use DataFrame.where/mask to avoid chained assignment quirks)
    mat = mat.mask(np.isfinite(mat) & (np.abs(mat) < 1e-12), 0.0)

    # 1) Order columns and compute blocks
    ordered_cols, col_blocks = compute_ordered_columns_and_blocks(list(mat.columns))
    mat2 = mat.reindex(columns=ordered_cols)

    # 2) Order rows by continent BEFORE plotting
    ordered_rows = order_rows_by_continent(mat2.index.tolist(), within_continent="alpha")
    mat2 = mat2.reindex(index=ordered_rows)

    # 3) Prepare data to plot (clip positives to 0 for negative-only shading)
    data_to_plot = mat2.clip(upper=0.0) if NEGATIVE_ONLY else mat2

    # 4) Compute color limits from the data we actually plot
    vals = data_to_plot.values.ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        vmin, vmax = (-1.0, 0.0)
    else:
        negs = vals[vals < 0]
        if negs.size == 0:
            vmin, vmax = (-1.0, 0.0)
        else:
            # robust lower bound (1st percentile); fallback to min if percentile >= 0 (rare)
            vmin = np.percentile(negs, 1.0)
            if vmin >= 0:
                vmin = float(np.min(negs))
            vmax = 0.0

    # 5) Plot
    cmap = plt.get_cmap("Blues_r").copy()
    cmap.set_bad(color="white", alpha=1.0)  # NaNs -> white

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(data_to_plot.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    # --- Row continent banding + separators + right labels (rows already ordered) ---
    row_blocks = _continent_blocks_for_rows(mat2.index.tolist())
    for idx, (cont, start, end) in enumerate(row_blocks):
        if idx % 2 == 0:
            ax.axhspan(start - 0.5, end + 0.5, facecolor="lightgrey", alpha=0.08, zorder=0)
    for k, (_, start, end) in enumerate(row_blocks):
        if k < len(row_blocks) - 1:
            ax.axhline(end + 0.5, color="black", lw=1.2, alpha=0.35)
    xmax = mat2.shape[1] - 0.5
    for cont, start, end in row_blocks:
        mid = (start + end) / 2.0
        ax.text(
            xmax + 0.15, mid, cont,
            va="center", ha="left", fontsize=10, color="dimgray", rotation=90, alpha=0.9, clip_on=False,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.6)
        )
    plt.subplots_adjust(right=0.92)

    # --- Column blocks: banding + separators + top labels ---
    _draw_column_group_decorations(ax, mat2, col_blocks)

    # ticks
    ax.set_xticks(np.arange(mat2.shape[1]))
    ax.set_xticklabels(mat2.columns, rotation=45, ha="right", fontsize=10)
    ax.set_yticks(np.arange(mat2.shape[0]))
    ax.set_yticklabels(mat2.index, fontsize=10)

    # faint grid
    for k in range(mat2.shape[1]-1):
        ax.axvline(k + 0.5, color="grey", lw=0.8, alpha=0.25)
    for k in range(mat2.shape[0]-1):
        ax.axhline(k + 0.5, color="grey", lw=0.8, alpha=0.2)

    ax.set_title(title, fontsize=14, pad=12)
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Absolute change in Solar PV (Million EUR)", rotation=90)

    # annotate (now consistent with the image)
    if annotate:
        H, W = mat2.shape
        mid_color_val = (vmin + vmax) / 2.0 if np.isfinite(vmin) else -0.5
        for i in range(H):
            for j in range(W):
                v = mat2.iat[i, j]
                if np.isfinite(v) and v != 0.0:
                    text_color = "white" if v < mid_color_val else "black"
                    ax.text(
                        j, i, annotation_fmt.format(v),
                        ha="center", va="center", fontsize=9, color=text_color,
                        bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none",
                                  alpha=0.35 if text_color == "black" else 0.15)
                    )

    fig.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close(fig)
    print(f"[Saved] {outpath}")


def _robust_neg_limits(arr, p=1.0, default=(-1.0, 0.0)):
    """
    Compute robust (vmin, vmax) for negatives in arr (numpy array),
    using p-th percentile for vmin and 0 for vmax. Returns default if no negatives.
    """
    vals = arr.ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return default
    negs = vals[vals < 0]
    if negs.size == 0:
        return default
    vmin = np.percentile(negs, p)
    if vmin >= 0:
        vmin = float(np.min(negs))
    return (vmin, 0.0)

def plot_heatmap_negative_only(matrix, title, outpath, figsize=(18, 10),
                               annotate=True, annotation_fmt="{:.3f}"):
    """
    Heatmap with continent grouping (rows) and 3-block column grouping.
    If SEPARATE_SCALES_BY_GROUP is True, each column group gets its own color scale.
    """
    mat = matrix.copy()
    # Clean tiny numerical noise to exactly 0
    mat = mat.mask(np.isfinite(mat) & (np.abs(mat) < 1e-12), 0.0)

    # --- Order columns + rows
    ordered_cols, col_blocks = compute_ordered_columns_and_blocks(list(mat.columns))
    mat2 = mat.reindex(columns=ordered_cols)

    ordered_rows = order_rows_by_continent(mat2.index.tolist(), within_continent="alpha")
    mat2 = mat2.reindex(index=ordered_rows)

    # Data to plot (mask positives so they appear white and don't influence scaling)
    base = mat2.where(mat2 < 0.0)

    # --- Figure/Axes
    fig, ax = plt.subplots(figsize=figsize)

    # --- Column blocks: banding + separators + top labels (before drawing images is fine)
    _draw_column_group_decorations(ax, mat2, col_blocks)

    # --- Row continent banding + separators + right labels
    row_blocks = _continent_blocks_for_rows(mat2.index.tolist())
    for idx, (cont, start, end) in enumerate(row_blocks):
        if idx % 2 == 0:
            ax.axhspan(start - 0.5, end + 0.5, facecolor="lightgrey", alpha=0.08, zorder=0)
    for k, (_, start, end) in enumerate(row_blocks):
        if k < len(row_blocks) - 1:
            ax.axhline(end + 0.5, color="black", lw=1.2, alpha=0.35)

    # ---- Draw images
    # We'll overlay one imshow per column block with its own vmin/vmax and its own colorbar.
    # Prepare a light-to-dark blues colormap with NaNs=white.
    base_cmap = plt.get_cmap("Blues_r").copy()
    base_cmap.set_bad(color="white", alpha=1.0)

    images = []
    cbar_axes = []
    divider = make_axes_locatable(ax)

    # Space on the right for 3 stacked colorbars
    # (increase right margin depending on label lengths)
    plt.subplots_adjust(right=0.82)

    # Place three colorbars to the right, top-to-bottom
    # Adjust pads if they overlap in your layout
    for bi, (gname, start, end) in enumerate(col_blocks):
        # Make a full-size masked array with NaN outside this block
        full = np.full(mat2.shape, np.nan, dtype=float)
        block = base.iloc[:, start:end+1]
        full[:, start:end+1] = block.values

        # Compute robust limits from the block values only
        vmin, vmax = _robust_neg_limits(block.values, p=1.0, default=(-1.0, 0.0))

        # Draw this block
        im = ax.imshow(full, aspect="auto", cmap=base_cmap, vmin=vmin, vmax=vmax)
        images.append((gname, im))

    # ticks and grid after the images
    ax.set_xticks(np.arange(mat2.shape[1]))
    ax.set_xticklabels(mat2.columns, rotation=45, ha="right", fontsize=10)
    ax.set_yticks(np.arange(mat2.shape[0]))
    ax.set_yticklabels(mat2.index, fontsize=10)

    for k in range(mat2.shape[1]-1):
        ax.axvline(k + 0.5, color="grey", lw=0.8, alpha=0.25)
    for k in range(mat2.shape[0]-1):
        ax.axhline(k + 0.5, color="grey", lw=0.8, alpha=0.2)

    # Right-side continent labels
    xmax = mat2.shape[1] - 0.5
    for cont, start, end in row_blocks:
        mid = (start + end) / 2.0
        ax.text(
            xmax + 0.15, mid, cont, va="center", ha="left",
            fontsize=10, color="dimgray", rotation=90, alpha=0.9, clip_on=False,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.6)
        )

    ax.set_title(title, fontsize=14, pad=12)

    # --- Add one colorbar per block (stacked on the right)
    # Create 3 cax with increasing pad so they don't overlap
    cax_pads = [0.02, 0.10, 0.18]
    for (gname, im), pad in zip(images, cax_pads):
        cax = divider.append_axes("right", size="2.6%", pad=pad)
        cb = fig.colorbar(im, cax=cax)
        cb.set_label(f"{gname}", rotation=90)
        cb.ax.tick_params(labelsize=8)
        cbar_axes.append(cax)

    # --- Annotate numeric values (three decimals)
    if annotate:
        H, W = mat2.shape
        # We pick the mid threshold from the overall most negative vmin used,
        # just to set the white/black text choice; you can refine per-block if desired.
        global_vmin = min([im.get_clim()[0] for _, im in images]) if images else -1.0
        mid_color_val = (global_vmin + 0.0) / 2.0
        for i in range(H):
            for j in range(W):
                v = mat2.iat[i, j]
                if np.isfinite(v) and v != 0.0:
                    text_color = "white" if v < mid_color_val else "black"
                    ax.text(
                        j, i, annotation_fmt.format(v),
                        ha="center", va="center", fontsize=9, color=text_color,
                        bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none",
                                  alpha=0.35 if text_color == "black" else 0.15)
                    )

    fig.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close(fig)
    print(f"[Saved] {outpath}")

    
# ==================== MAIN ====================
if __name__ == "__main__":
    # Load master CSV
    master = pd.read_csv(MASTER_CSV)

    # Detect destinations
    detected_destinations = infer_destination_list(master.columns)
    if DEST_LIST is None:
        DEST_LIST = detected_destinations
    else:
        DEST_LIST = [d for d in DEST_LIST if f"abs_{d}_EUR" in master.columns]

    if not DEST_LIST:
        raise ValueError("No destination columns found in master CSV (abs_<CC>_EUR).")

    # Required columns
    for req in ("Shock_Industry","Shock_Country"):
        if req not in master.columns:
            raise ValueError(f"Master CSV missing column '{req}'.")

    # For stable column ordering: preserve master industry order
    industries_order = list(pd.Index(master["Shock_Industry"]).unique())

    # Per-destination rendering
    for dest in DEST_LIST:
        # Build matrix from master
        mat_dest = build_matrix_from_master(
            master_df=master,
            dest_country=dest,
            top_n_rows=TOP_COUNTRIES_FOR_HEATMAP
        )

        # Ensure columns appear as in master (then grouped)
        # (Not strictly necessary: compute_ordered_columns_and_blocks reorders anyway)
        mat_dest = mat_dest.reindex(columns=[c for c in industries_order if c in mat_dest.columns])

        # Output paths
        out_dir = os.path.join(OUT_DIR_BASE, dest)
        os.makedirs(out_dir, exist_ok=True)
        out_png = os.path.join(out_dir, f"heatmap_abs_changes_negonly_white0_{dest}.png")
        out_csv = os.path.join(out_dir, f"matrix_{dest}.csv")

        # Title
        title = (
            "Absolute change in Solar PV (Million EUR; 0=white → more negative=darker blue) "
            f"— Target: {dest}\n"
            f"Rows: shock countries ({'Top '+str(TOP_COUNTRIES_FOR_HEATMAP) if TOP_COUNTRIES_FOR_HEATMAP else 'All'}, grouped by continent), "
            f"Cols: shock industries (tertiary → secondary → primary)"
        )

        # Plot & save
        plot_heatmap_negative_only(
            matrix=mat_dest,
            title=title,
            outpath=out_png,
            figsize=FIGSIZE,
            annotate=ANNOTATE,
            annotation_fmt=ANNOTATION_FMT
        )

        # Save the numeric matrix used
        mat_dest.to_csv(out_csv)
        print(f"[Saved] {out_csv}")


[Saved] heatmaps_from_master\AT\heatmap_abs_changes_negonly_white0_AT.png
[Saved] heatmaps_from_master\AT\matrix_AT.csv
[Saved] heatmaps_from_master\BE\heatmap_abs_changes_negonly_white0_BE.png
[Saved] heatmaps_from_master\BE\matrix_BE.csv
[Saved] heatmaps_from_master\BG\heatmap_abs_changes_negonly_white0_BG.png
[Saved] heatmaps_from_master\BG\matrix_BG.csv
[Saved] heatmaps_from_master\CY\heatmap_abs_changes_negonly_white0_CY.png
[Saved] heatmaps_from_master\CY\matrix_CY.csv
[Saved] heatmaps_from_master\CZ\heatmap_abs_changes_negonly_white0_CZ.png
[Saved] heatmaps_from_master\CZ\matrix_CZ.csv
[Saved] heatmaps_from_master\DE\heatmap_abs_changes_negonly_white0_DE.png
[Saved] heatmaps_from_master\DE\matrix_DE.csv
[Saved] heatmaps_from_master\DK\heatmap_abs_changes_negonly_white0_DK.png
[Saved] heatmaps_from_master\DK\matrix_DK.csv
[Saved] heatmaps_from_master\EE\heatmap_abs_changes_negonly_white0_EE.png
[Saved] heatmaps_from_master\EE\matrix_EE.csv
[Saved] heatmaps_from_master\ES\heatmap_

In [14]:
# works well!!! with group headings
# -*- coding: utf-8 -*-
"""
Create per-destination heatmaps from the 'master' CSV produced by the Ghosh shock sweep.

Master CSV columns expected:
  - 'Shock_Industry', 'Shock_Country', 'Shock_Output_EUR'
  - one column per destination: 'abs_<CC>_EUR' (absolute change in Solar PV for <CC>, Million EUR)

Output folder structure:
  heatmaps_from_master/
    <DEST>/
      heatmap_dependence_shares_<DEST>.png
      matrix_abs_impacts_<DEST>.csv
      matrix_dependence_shares_<DEST>.csv

Shading encodes 'percentage of dependence' (column share) for each industry:
  share_ij = |impact_ij| / sum_k |impact_kj|
  (by default using negative impacts only; configurable)
"""

# ==================== IMPORTS ====================
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- Visual defaults ----------------
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

# ==================== CONFIG ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv"  # <-- set actual path
OUT_DIR_BASE = "heatmaps_from_master"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

# If None → all destinations found in master; or pass a list like ["CN","FR","DE","JP","US"]
DEST_LIST = None

# Rows in the heatmap (choose ~15–30 for readability). If None -> keep all.
TOP_COUNTRIES_FOR_HEATMAP = 49

FIGSIZE = (18, 10)   # per-destination heatmap size

# --- Shares computation settings ---
# True -> shares computed from negative impacts only (positives ignored); False -> use absolute of all impacts
SHARE_NEGATIVE_ONLY = True

# Color scaling for shares (0..1). Keep fixed=1.0 for comparability across destinations.
SHARE_COLORBAR_MAX = 1.0

# --- Column groups (similar to your example: tertiary -> secondary -> primary) ---
PRIORITY_FIRST = [
    "Electrical machinery and apparatus n.e.c. (31)",
    "Fabricated metal products, except machinery and equipment (28)",
    "Machinery and equipment n.e.c. (29)",
]
METAL_PRODUCTS_BLOCK = [
    "Aluminium and aluminium products",
    "Basic iron and steel and of ferro-alloys and first products thereof",
    "Copper products",
    "Lead, zinc and tin and products thereof",
    "Other non-ferrous metal products",
    "Precious metals",
    # If you want to include the ores here, move them to the 'rest' set or add another block
    # "Precious metal ores and concentrates",  # typically primary, so keep in 'rest'
]

# --- Continent mapping (extend to your code set; supports 2- or 3-letter codes) ---
CONTINENT_MAP = {
    # Europe (2-letter)
    "AT":"Europe","BE":"Europe","BG":"Europe","CH":"Europe","CY":"Europe","CZ":"Europe","DE":"Europe","DK":"Europe","EE":"Europe",
    "ES":"Europe","FI":"Europe","FR":"Europe","GB":"Europe","GR":"Europe","HR":"Europe","HU":"Europe",
    "IE":"Europe","IT":"Europe","LT":"Europe","LU":"Europe","LV":"Europe","MT":"Europe","NL":"Europe","NO":"Europe","PL":"Europe","PT":"Europe","RO":"Europe",
    "RU":"Europe","SE":"Europe","SI":"Europe","SK":"Europe","WE":"Europe",
    # Americas (2-letter)
    "BR":"Americas","CA":"Americas","MX":"Americas","US":"Americas","WL":"Americas",
    # Asia (2-letter)
    "CN":"Asia","ID":"Asia","IN":"Asia","JP":"Asia","KR":"Asia","TW":"Asia","TR":"Asia","WA":"Asia",
    # Oceania
    "AU":"Oceania",
    # Africa
    "ZA":"Africa","WF":"Africa",
    # Middle East
    "WM":"Middle East",
}
CONTINENT_ORDER = ["Europe", "Americas", "Asia", "Middle East", "Africa", "Oceania"]

# ==================== HELPERS ====================
def _norm(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s)).strip().lower()
    s = s.replace("&amp;amp;", " and ")
    s = s.replace("aluminium", "aluminum")  # unify spelling if needed for tokens
    s = s.replace("photovoltaics", "photovoltaic")
    return s

def classify_industry(ind_name: str):
    """
    Return one of {'tertiary','secondary','primary'} via simple token search.
    This is best-effort; we enforce ordering via explicit lists below.
    """
    s = _norm(ind_name)
    if any(k in s for k in ["machinery","apparatus","equipment","n.e.c"]):
        return "tertiary"
    if any(k in s for k in ["product","basic","precious","non-ferrous product"]):
        return "secondary"
    if any(k in s for k in ["ore","concentrate","mining"]):
        return "primary"
    # fallback
    return "tertiary"

def compute_ordered_columns_and_blocks(present_cols):
    """
    Return ordered_cols and column group blocks aligned to present_cols:
      ordered_cols: [priority_first (present)] + [metal_products (present)] + [rest alphabetical]
      col_blocks: list of (group_name, start_idx, end_idx)
    """
    present_set = set(present_cols)
    priority = [c for c in PRIORITY_FIRST if c in present_set]
    metals   = [c for c in METAL_PRODUCTS_BLOCK if c in present_set]
    rest     = sorted([c for c in present_cols if c not in set(PRIORITY_FIRST + METAL_PRODUCTS_BLOCK)])

    ordered_cols = priority + metals + rest

    # blocks for banding/labels
    blocks = []
    start = 0
    if len(priority) > 0:
        blocks.append(("Tertiary", start, start + len(priority) - 1))
        start += len(priority)
    if len(metals) > 0:
        blocks.append(("Secondary", start, start + len(metals) - 1))
        start += len(metals)
    if len(rest) > 0:
        # Heuristic: assume remaining 'rest' are primary (mostly ores & concentrates)
        blocks.append(("Primary", start, start + len(rest) - 1))

    return ordered_cols, blocks

def infer_destination_list(master_columns):
    """Return list of destination country codes from 'abs_<CC>_EUR' columns."""
    dests = []
    for c in master_columns:
        if isinstance(c, str) and c.startswith("abs_") and c.endswith("_EUR"):
            dests.append(c[len("abs_"):-len("_EUR")])
    return dests

def order_rows_by_continent(rows, within_continent="alpha"):
    """
    Reorder the given 'rows' (country codes) by CONTINENT_ORDER.
    within_continent: 'alpha' or 'keep'
    """
    buckets = {cont: [] for cont in CONTINENT_ORDER}
    others = []
    for r in rows:
        cont = CONTINENT_MAP.get(str(r))
        if cont in buckets:
            buckets[cont].append(r)
        else:
            others.append(r)
    if within_continent == "alpha":
        for cont in buckets:
            buckets[cont].sort()
    ordered = []
    for cont in CONTINENT_ORDER:
        ordered.extend(buckets[cont])
    ordered.extend(others)
    return ordered

def _continent_blocks_for_rows(rows):
    """
    Given an ordered list of rows (country codes), return
    (continent_name, start_idx, end_idx) for contiguous blocks.
    """
    blocks = []
    i = 0
    n = len(rows)
    while i < n:
        cont = CONTINENT_MAP.get(str(rows[i]), "Other")
        start = i
        i += 1
        while i < n and CONTINENT_MAP.get(str(rows[i]), "Other") == cont:
            i += 1
        end = i - 1
        blocks.append((cont, start, end))
    return blocks

def _draw_column_group_decorations(ax, mat2, col_blocks):
    """Add vertical banding, separators, and top labels for column groups."""
    # light banding
    for idx, (gname, start, end) in enumerate(col_blocks):
        if idx % 2 == 0:
            ax.axvspan(start - 0.5, end + 0.5, facecolor="lightgrey", alpha=0.08, zorder=0)
    # separators
    for k, (_, start, end) in enumerate(col_blocks):
        if k < len(col_blocks) - 1:
            ax.axvline(end + 0.5, color="black", lw=1.2, alpha=0.35)
    # top labels
    ymax = -0.5
    for gname, start, end in col_blocks:
        mid = (start + end) / 2.0
        ax.text(
            mid, ymax - 0.15, gname,
            va="top", ha="center", fontsize=10, color="dimgray", rotation=0, alpha=0.95, clip_on=False,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.6),
            transform=ax.transData
        )
    plt.subplots_adjust(top=0.88)

def save_dataframe_rounded(df: pd.DataFrame, path: str, decimals: int = 3):
    """Save DataFrame rounded to 'decimals' as CSV."""
    df_round = df.copy()
    with np.errstate(invalid='ignore'):
        df_round = df_round.round(decimals)
    df_round.to_csv(path)
    print(f"[Saved] {path} (rounded to {decimals} decimals)")

# ==================== CORE BUILDERS ====================
def build_matrix_from_master(master_df, dest_country, top_n_rows=None):
    """
    From the master CSV, build the matrix:
      rows = Shock_Country
      cols = Shock_Industry
      values = abs change for dest_country (Million EUR)
    Apply Top-N filter by aggregated |impact| across industries if requested.
    """
    val_col = f"abs_{dest_country}_EUR"
    if val_col not in master_df.columns:
        raise KeyError(f"Destination column '{val_col}' not found in master CSV.")

    df = master_df[["Shock_Country", "Shock_Industry", val_col]].copy()
    df.rename(columns={val_col: "value"}, inplace=True)

    # Pivot
    mat = df.pivot(index="Shock_Country", columns="Shock_Industry", values="value")

    # Top-N rows by aggregate magnitude
    if top_n_rows is not None and top_n_rows > 0:
        mag = mat.abs().sum(axis=1).sort_values(ascending=False)
        keep_rows = list(mag.head(top_n_rows).index)
        mat = mat.loc[keep_rows, :]

    return mat

# ==================== PLOTTING (shares shading, no annotations) ====================
def plot_heatmap_dependence_shares(matrix, title, outpath, figsize=(18, 10)):
    """
    Shade cells by 'percentage of dependence' (column share in [0,1]).
    - Rows grouped by continent, column groups: Tertiary -> Secondary -> Primary.
    - Single color scale (0=white, 1=dark blue).
    - No numeric annotations in cells.
    """
    mat = matrix.copy()
    # Clean tiny numerical noise to exactly 0
    mat = mat.mask(np.isfinite(mat) & (np.abs(mat) < 1e-12), 0.0)

    # --- Order columns + rows
    ordered_cols, col_blocks = compute_ordered_columns_and_blocks(list(mat.columns))
    mat2 = mat.reindex(columns=ordered_cols)

    ordered_rows = order_rows_by_continent(mat2.index.tolist(), within_continent="alpha")
    mat2 = mat2.reindex(index=ordered_rows)

    # --- Build shares matrix (per column normalization)
    if SHARE_NEGATIVE_ONLY:
        # Consider only negative impacts; positives -> NaN (ignored in sums)
        base = mat2.where(mat2 < 0.0)
        numer = base.abs()
    else:
        # Use absolute of all impacts (negatives + positives)
        numer = mat2.abs()

    denom = numer.sum(axis=0)  # per-column sum
    # Avoid division warnings; columns with denom=0 become all-NaN (white cells)
    shares = numer.divide(denom.where(denom != 0), axis=1)

    # --- Plot
    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad(color="white", alpha=1.0)  # NaNs -> white

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(shares.values, aspect="auto", cmap=cmap, vmin=0.0, vmax=float(SHARE_COLORBAR_MAX))

    # Row continent banding + separators + right labels
    row_blocks = _continent_blocks_for_rows(mat2.index.tolist())
    for idx, (cont, start, end) in enumerate(row_blocks):
        if idx % 2 == 0:
            ax.axhspan(start - 0.5, end + 0.5, facecolor="lightgrey", alpha=0.08, zorder=0)
    for k, (_, start, end) in enumerate(row_blocks):
        if k < len(row_blocks) - 1:
            ax.axhline(end + 0.5, color="black", lw=1.2, alpha=0.35)
    xmax = mat2.shape[1] - 0.5
    for cont, start, end in row_blocks:
        mid = (start + end) / 2.0
        ax.text(
            xmax + 0.15, mid, cont,
            va="center", ha="left", fontsize=10, color="dimgray", rotation=90, alpha=0.9, clip_on=False,
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.6)
        )
    plt.subplots_adjust(right=0.92)

    # Column blocks: banding + separators + top labels
    _draw_column_group_decorations(ax, mat2, col_blocks)

    # ticks
    ax.set_xticks(np.arange(mat2.shape[1]))
    ax.set_xticklabels(mat2.columns, rotation=45, ha="right", fontsize=10)
    ax.set_yticks(np.arange(mat2.shape[0]))
    ax.set_yticklabels(mat2.index, fontsize=10)

    # faint grid
    for k in range(mat2.shape[1]-1):
        ax.axvline(k + 0.5, color="grey", lw=0.8, alpha=0.25)
    for k in range(mat2.shape[0]-1):
        ax.axhline(k + 0.5, color="grey", lw=0.8, alpha=0.2)

    ax.set_title(title, fontsize=14, pad=12)
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Industry Specific Source Country Dependence Share", rotation=90)

    fig.tight_layout()
    plt.savefig(outpath)
    plt.close(fig)
    print(f"[Saved] {outpath}")

    return shares  # return shares matrix for optional saving upstream

# ==================== MAIN ====================
if __name__ == "__main__":
    # Load master CSV
    if not os.path.exists(MASTER_CSV):
        raise FileNotFoundError(f"MASTER_CSV not found: {MASTER_CSV}")

    print(f"[Loading master] {MASTER_CSV}")
    master = pd.read_csv(MASTER_CSV)

    # Detect destinations
    detected_destinations = infer_destination_list(master.columns)
    if DEST_LIST is None:
        DEST_LIST = detected_destinations
    else:
        DEST_LIST = [d for d in DEST_LIST if f"abs_{d}_EUR" in master.columns]

    if not DEST_LIST:
        raise ValueError("No destination columns found in master CSV (abs_<CC>_EUR).")

    print(f"[Destinations] {len(DEST_LIST)} detected: {DEST_LIST}")

    # Required columns
    for req in ("Shock_Industry","Shock_Country"):
        if req not in master.columns:
            raise ValueError(f"Master CSV missing column '{req}'.")

    # For stable column ordering: preserve master industry order
    industries_order = list(pd.Index(master["Shock_Industry"]).unique())

    # Per-destination rendering
    total = len(DEST_LIST)
    for idx, dest in enumerate(DEST_LIST, start=1):
        print(f"\n[Progress] {idx}/{total} → {dest}")

        # Build matrix from master (absolute impacts)
        mat_dest = build_matrix_from_master(
            master_df=master,
            dest_country=dest,
            top_n_rows=TOP_COUNTRIES_FOR_HEATMAP
        )

        if mat_dest.empty:
            print(f"[Skip] Empty matrix for destination {dest}")
            continue

        # Ensure columns appear as in master (then grouped)
        mat_dest = mat_dest.reindex(columns=[c for c in industries_order if c in mat_dest.columns])

        # Output paths
        out_dir = os.path.join(OUT_DIR_BASE, dest)
        os.makedirs(out_dir, exist_ok=True)
        out_png = os.path.join(out_dir, f"heatmap_dependence_shares_{dest}.png")
        out_csv_abs = os.path.join(out_dir, f"matrix_abs_impacts_{dest}.csv")
        out_csv_share = os.path.join(out_dir, f"matrix_dependence_shares_{dest}.csv")

        # Title
        title = (
            f"{dest} - "
            "Solar Energy Supply Chain Dependence"
            f"\n Rows: shock countries (grouped by continent), "
            f"Cols: shock industries (grouped by production stage). "
            f"\n Shading: column shares of |negative impacts|"
            if SHARE_NEGATIVE_ONLY else
            f"{dest} - Solar Energy Supply Chain Dependence\n Rows grouped by continent; "
            f"Cols grouped by production stage.\n Shading: column shares of |all impacts|"
        )

        # Plot (shades by shares) & save
        shares_dest = plot_heatmap_dependence_shares(
            matrix=mat_dest,
            title=title,
            outpath=out_png,
            figsize=FIGSIZE
        )

        # Save the numeric matrices (rounded for readability)
        save_dataframe_rounded(mat_dest, out_csv_abs, decimals=3)      # absolute impacts used as base
        save_dataframe_rounded(shares_dest, out_csv_share, decimals=3) # shares used for shading

[Loading master] ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv
[Destinations] 49 detected: ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'GB', 'US', 'JP', 'CN', 'CA', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'CH', 'TR', 'TW', 'NO', 'ID', 'ZA', 'WA', 'WL', 'WE', 'WF', 'WM']

[Progress] 1/49 → AT
[Saved] heatmaps_from_master\AT\heatmap_dependence_shares_AT.png
[Saved] heatmaps_from_master\AT\matrix_abs_impacts_AT.csv (rounded to 3 decimals)
[Saved] heatmaps_from_master\AT\matrix_dependence_shares_AT.csv (rounded to 3 decimals)

[Progress] 2/49 → BE
[Saved] heatmaps_from_master\BE\heatmap_dependence_shares_BE.png
[Saved] heatmaps_from_master\BE\matrix_abs_impacts_BE.csv (rounded to 3 decimals)
[Saved] heatmaps_from_master\BE\matrix_dependence_shares_BE.csv (rounded to 3 decimals)

[Progress] 3/49 → BG
[Saved] heatmaps_from_master\BG\heatmap_de

In [15]:
#final

# -*- coding: utf-8 -*-
"""
Create per-destination heatmaps from the 'master' CSV produced by the Ghosh shock sweep.

Master CSV columns expected:
  - 'Shock_Industry', 'Shock_Country', 'Shock_Output_EUR'
  - one column per destination: 'abs_<CC>_EUR' (absolute change in Solar PV for <CC>, Million EUR)

Output folder structure:
  heatmaps_from_master/
    <DEST>/
      heatmap_dependence_shares_<DEST>.png
      matrix_abs_impacts_<DEST>.csv
      matrix_dependence_shares_<DEST>.csv

Shading encodes 'percentage of dependence' (column share):
  share_ij = |impact_ij| / sum_k |impact_kj|
  (by default using negative impacts only; configurable)

This version:
  - Annotates TOP_K_PER_COLUMN largest shares per column (3 decimals).
  - Keeps faint cell gridlines and strong group separators.
  - Removes group headers (no continent or stage names).
  - Wraps long x-axis industry labels and staggers every second label lower.
  - Makes ONLY every second x-tick longer (to visually reach the lowered labels).
"""
# ---------------- Visual defaults ----------------
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

# ==================== CONFIG ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv"  # <-- set actual path
OUT_DIR_BASE = "heatmaps_from_master"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

# If None → all destinations found in master; or pass a list like ["CN","FR","DE","JP","US"]
DEST_LIST = None

# Rows in the heatmap (choose ~15–30 for readability). If None -> keep all.
TOP_COUNTRIES_FOR_HEATMAP = 49

FIGSIZE = (18, 10)   # default per-destination heatmap size

# --- Shares computation settings ---
# True -> shares computed from negative impacts only (positives ignored); False -> use absolute of all impacts
SHARE_NEGATIVE_ONLY = True

# Color scaling for shares (0..1). Keep fixed=1.0 for comparability across destinations.
SHARE_COLORBAR_MAX = 1.0

# --- Annotation settings (TOP-K per column) ---
ANNOTATE = True
TOP_K_PER_COLUMN = 5        # annotate top-K per column
ANNOTATION_DECIMALS = 3
DARK_BG_TEXT_THRESHOLD = 0.5  # use white text if share >= this

# --- Group separator settings (strong lines between blocks) ---
DRAW_GROUP_SEPARATORS = True
SEP_COLOR = "black"
SEP_ALPHA = 0.6
SEP_LW = 1.8  # line width for group separators

# --- Cell gridlines (faint) ---
DRAW_CELL_GRID = True
GRID_V_COLOR = "grey"
GRID_H_COLOR = "grey"
GRID_V_ALPHA = 0.25
GRID_H_ALPHA = 0.20
GRID_V_LW = 0.8
GRID_H_LW = 0.8

# --- X-axis label handling (wrap + stagger) ---
WRAP_WIDTH = 26
SHIFT_EVERY_SECOND_LABEL = True
SHIFT_LABEL_EXTRA_LINES = 3    # number of blank lines to prepend to every second label

# --- X-tick styling (only every second tick longer) ---
XTICK_PAD           = 6               # space between tick and label
XTICK_LEN_SHORT     = 6               # length for 'normal' ticks
XTICK_LEN_LONG      = 30              # length for the staggered ticks (every second)
XTICK_WIDTH_SHORT   = 1.0
XTICK_WIDTH_LONG    = 1.6

# --- Column groups (ordering only; we do not draw headers) ---
PRIORITY_FIRST = [
    "Electrical machinery and apparatus n.e.c. (31)",
    "Fabricated metal products, except machinery and equipment (28)",
    "Machinery and equipment n.e.c. (29)",
]
METAL_PRODUCTS_BLOCK = [
    "Aluminium and aluminium products",
    "Basic iron and steel and of ferro-alloys and first products thereof",
    "Copper products",
    "Lead, zinc and tin and products thereof",
    "Other non-ferrous metal products",
    "Precious metals",
]

# --- Continent mapping (ordering only; we do not draw headers) ---
CONTINENT_MAP = {
    # Europe (2-letter)
    "AT":"Europe","BE":"Europe","BG":"Europe","CH":"Europe","CY":"Europe","CZ":"Europe","DE":"Europe","DK":"Europe","EE":"Europe",
    "ES":"Europe","FI":"Europe","FR":"Europe","GB":"Europe","GR":"Europe","HR":"Europe","HU":"Europe",
    "IE":"Europe","IT":"Europe","LT":"Europe","LU":"Europe","LV":"Europe","MT":"Europe","NL":"Europe","NO":"Europe","PL":"Europe","PT":"Europe","RO":"Europe",
    "RU":"Europe","SE":"Europe","SI":"Europe","SK":"Europe","WE":"Europe",
    # Americas (2-letter)
    "BR":"Americas","CA":"Americas","MX":"Americas","US":"Americas","WL":"Americas",
    # Asia (2-letter)
    "CN":"Asia","ID":"Asia","IN":"Asia","JP":"Asia","KR":"Asia","TW":"Asia","TR":"Asia","WA":"Asia",
    # Oceania
    "AU":"Oceania",
    # Africa
    "ZA":"Africa","WF":"Africa",
    # Middle East
    "WM":"Middle East",
}
CONTINENT_ORDER = ["Europe", "Americas", "Asia", "Middle East", "Africa", "Oceania"]

# ==================== HELPERS ====================
def _norm(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s)).strip().lower()
    s = s.replace("&amp;amp;", " and ")
    s = s.replace("aluminium", "aluminum")  # unify spelling in tokens
    s = s.replace("photovoltaics", "photovoltaic")
    return s

def classify_industry(ind_name: str):
    """
    Return one of {'tertiary','secondary','primary'} via simple token search.
    Best-effort; ordering enforced via explicit lists below.
    """
    s = _norm(ind_name)
    if any(k in s for k in ["machinery","apparatus","equipment","n.e.c"]):
        return "tertiary"
    if any(k in s for k in ["product","basic","precious","non-ferrous product"]):
        return "secondary"
    if any(k in s for k in ["ore","concentrate","mining"]):
        return "primary"
    # fallback
    return "tertiary"

def compute_ordered_columns_and_blocks(present_cols):
    """
    Return ordered_cols and column group blocks aligned to present_cols:
      ordered_cols: [priority_first (present)] + [metal_products (present)] + [rest alphabetical]
      col_blocks: list of (group_name, start_idx, end_idx)
    """
    present_set = set(present_cols)
    priority = [c for c in PRIORITY_FIRST if c in present_set]
    metals   = [c for c in METAL_PRODUCTS_BLOCK if c in present_set]
    rest     = sorted([c for c in present_cols if c not in set(PRIORITY_FIRST + METAL_PRODUCTS_BLOCK)])

    ordered_cols = priority + metals + rest

    # blocks for placing strong separators (we won’t draw headers)
    blocks = []
    start = 0
    if len(priority) > 0:
        blocks.append(("Tertiary", start, start + len(priority) - 1))
        start += len(priority)
    if len(metals) > 0:
        blocks.append(("Secondary", start, start + len(metals) - 1))
        start += len(metals)
    if len(rest) > 0:
        blocks.append(("Primary", start, start + len(rest) - 1))

    return ordered_cols, blocks

def infer_destination_list(master_columns):
    """Return list of destination country codes from 'abs_<CC>_EUR' columns."""
    dests = []
    for c in master_columns:
        if isinstance(c, str) and c.startswith("abs_") and c.endswith("_EUR"):
            dests.append(c[len("abs_"):-len("_EUR")])
    return dests

def order_rows_by_continent(rows, within_continent="alpha"):
    """
    Reorder the given 'rows' (country codes) by CONTINENT_ORDER.
    within_continent: 'alpha' or 'keep'
    """
    buckets = {cont: [] for cont in CONTINENT_ORDER}
    others = []
    for r in rows:
        cont = CONTINENT_MAP.get(str(r))
        if cont in buckets:
            buckets[cont].append(r)
        else:
            others.append(r)
    if within_continent == "alpha":
        for cont in buckets:
            buckets[cont].sort()
    ordered = []
    for cont in CONTINENT_ORDER:
        ordered.extend(buckets[cont])
    ordered.extend(others)
    return ordered

def _continent_blocks_for_rows(rows):
    """
    Given an ordered list of rows (country codes), return
    (continent_name, start_idx, end_idx) for contiguous blocks.
    """
    blocks = []
    i = 0
    n = len(rows)
    while i < n:
        cont = CONTINENT_MAP.get(str(rows[i]), "Other")
        start = i
        i += 1
        while i < n and CONTINENT_MAP.get(str(rows[i]), "Other") == cont:
            i += 1
        end = i - 1
        blocks.append((cont, start, end))
    return blocks

def _wrap_label(s, width=WRAP_WIDTH):
    """Wrap long x-axis labels to multiple lines."""
    return "\n".join(textwrap.wrap(str(s), width=width))

# ==================== CORE BUILDERS ====================
def build_matrix_from_master(master_df, dest_country, top_n_rows=None):
    """
    From the master CSV, build the matrix:
      rows = Shock_Country
      cols = Shock_Industry
      values = abs change for dest_country (Million EUR)
    Apply Top-N filter by aggregated |impact| across industries if requested.
    """
    val_col = f"abs_{dest_country}_EUR"
    if val_col not in master_df.columns:
        raise KeyError(f"Destination column '{val_col}' not found in master CSV.")

    df = master_df[["Shock_Country", "Shock_Industry", val_col]].copy()
    df.rename(columns={val_col: "value"}, inplace=True)

    # Pivot
    mat = df.pivot(index="Shock_Country", columns="Shock_Industry", values="value")

    # Top-N rows by aggregate magnitude
    if top_n_rows is not None and top_n_rows > 0:
        mag = mat.abs().sum(axis=1).sort_values(ascending=False)
        keep_rows = list(mag.head(top_n_rows).index)
        mat = mat.loc[keep_rows, :]

    return mat

def save_dataframe_rounded(df: pd.DataFrame, path: str, decimals: int = 3):
    """Save DataFrame rounded to 'decimals' as CSV."""
    df_round = df.copy()
    with np.errstate(invalid='ignore'):
        df_round = df_round.round(decimals)
    df_round.to_csv(path)
    print(f"[Saved] {path} (rounded to {decimals} decimals)")

# ==================== PLOTTING (no headers; wrap+stagger labels; alt tick lengths; gridlines; strong separators; top-5 per column) ====================
def plot_heatmap_dependence_shares(matrix, title, outpath, figsize=(18, 10)):
    mat = matrix.copy()
    mat = mat.mask(np.isfinite(mat) & (np.abs(mat) < 1e-12), 0.0)

    # Column ordering + blocks
    ordered_cols, col_blocks = compute_ordered_columns_and_blocks(list(mat.columns))
    mat2 = mat.reindex(columns=ordered_cols)

    # Row ordering + continent blocks
    ordered_rows = order_rows_by_continent(mat2.index.tolist(), within_continent="alpha")
    mat2 = mat2.reindex(index=ordered_rows)
    row_blocks = _continent_blocks_for_rows(mat2.index.tolist())

    # Shares
    if SHARE_NEGATIVE_ONLY:
        base = mat2.where(mat2 < 0.0)
        numer = base.abs()
    else:
        numer = mat2.abs()
    denom = numer.sum(axis=0)
    shares = numer.divide(denom.where(denom != 0), axis=1)  # columns with denom=0 -> NaN (white)

    # Plot
    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad(color="white", alpha=1.0)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(shares.values, aspect="auto", cmap=cmap, vmin=0.0, vmax=float(SHARE_COLORBAR_MAX))

    # Faint per-cell gridlines
    if DRAW_CELL_GRID:
        for k in range(mat2.shape[1]-1):
            ax.axvline(k + 0.5, color=GRID_V_COLOR, lw=GRID_V_LW, alpha=GRID_V_ALPHA, zorder=2)
        for k in range(mat2.shape[0]-1):
            ax.axhline(k + 0.5, color=GRID_H_COLOR, lw=GRID_H_LW, alpha=GRID_H_ALPHA, zorder=2)

    # Strong separators between groups (columns: industry blocks; rows: continent blocks)
    if DRAW_GROUP_SEPARATORS:
        for k, (_, start, end) in enumerate(col_blocks):
            if k < len(col_blocks) - 1:
                ax.axvline(end + 0.5, color=SEP_COLOR, lw=SEP_LW, alpha=SEP_ALPHA, zorder=3)
        for k, (_, start, end) in enumerate(row_blocks):
            if k < len(row_blocks) - 1:
                ax.axhline(end + 0.5, color=SEP_COLOR, lw=SEP_LW, alpha=SEP_ALPHA, zorder=3)

    # ---------------- TICKS & LABELS (wrap + stagger every second; only every second tick longer) ----------------
    # Build wrapped labels
    wrapped_labels = [_wrap_label(c, width=WRAP_WIDTH) for c in mat2.columns]

    # Stagger every second label by prepending blank lines
    if SHIFT_EVERY_SECOND_LABEL and SHIFT_LABEL_EXTRA_LINES > 0:
        pad_lines = "\n" * SHIFT_LABEL_EXTRA_LINES
        for j in range(len(wrapped_labels)):
            if j % 2 == 1:
                wrapped_labels[j] = pad_lines + wrapped_labels[j]

    # Apply ticks & labels
    ax.set_xticks(np.arange(mat2.shape[1]))
    ax.set_xticklabels(wrapped_labels, rotation=0, ha="center", fontsize=9)
    ax.set_yticks(np.arange(mat2.shape[0]))
    ax.set_yticklabels(mat2.index, fontsize=10)

    # Default tick params (short ticks for all; 'out' direction)
    ax.tick_params(axis='x', which='major', length=XTICK_LEN_SHORT, width=XTICK_WIDTH_SHORT, direction='out', pad=XTICK_PAD)
    ax.tick_params(axis='y', which='major', length=6, width=1.0, direction='out')

    # Make ONLY every second bottom tick longer (aligning to lowered labels)
    # Each tick object has tick1line (bottom) and tick2line (top). We extend only tick1line.
    for j, t in enumerate(ax.xaxis.get_major_ticks()):
        if j % 2 == 1:
            t.tick1line.set_markersize(XTICK_LEN_LONG)   # longer bottom tick
            t.tick1line.set_linewidth(XTICK_WIDTH_LONG)  # thicker bottom tick
        else:
            t.tick1line.set_markersize(XTICK_LEN_SHORT)
            t.tick1line.set_linewidth(XTICK_WIDTH_SHORT)

    # Give a bit more room at the bottom for the staggered labels
    plt.subplots_adjust(bottom=0.24)

    # Annotate TOP_K_PER_COLUMN largest shares per column (3 decimals)
    if ANNOTATE:
        nrows, ncols = shares.shape
        for j in range(ncols):
            col = shares.iloc[:, j]
            top_idx = col.dropna().nlargest(TOP_K_PER_COLUMN).index  # handles columns with <K non-NaN
            for row_label in top_idx:
                i = shares.index.get_loc(row_label)
                val = shares.iat[i, j]
                txt_color = "white" if val >= DARK_BG_TEXT_THRESHOLD else "black"
                ax.text(
                    j, i, f"{val:.{ANNOTATION_DECIMALS}f}",
                    ha="center", va="center",
                    fontsize=9, color=txt_color, zorder=4
                )

    ax.set_title(title, fontsize=14, pad=12)
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Industry-specific source-country dependence share", rotation=90)

    # Save
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[Saved] {outpath}")

    return shares

# ==================== MAIN ====================
if __name__ == "__main__":
    # Load master CSV
    if not os.path.exists(MASTER_CSV):
        raise FileNotFoundError(f"MASTER_CSV not found: {MASTER_CSV}")

    print(f"[Loading master] {MASTER_CSV}")
    master = pd.read_csv(MASTER_CSV)

    # Detect destinations
    detected_destinations = infer_destination_list(master.columns)
    if DEST_LIST is None:
        DEST_LIST = detected_destinations
    else:
        DEST_LIST = [d for d in DEST_LIST if f"abs_{d}_EUR" in master.columns]

    if not DEST_LIST:
        raise ValueError("No destination columns found in master CSV (abs_<CC>_EUR).")

    print(f"[Destinations] {len(DEST_LIST)} detected: {DEST_LIST}")

    # Required columns
    for req in ("Shock_Industry","Shock_Country"):
        if req not in master.columns:
            raise ValueError(f"Master CSV missing column '{req}'.")

    # For stable column ordering: preserve master industry order
    industries_order = list(pd.Index(master["Shock_Industry"]).unique())

    # Per-destination rendering
    total = len(DEST_LIST)
    for idx, dest in enumerate(DEST_LIST, start=1):
        print(f"\n[Progress] {idx}/{total} → {dest}")

        # Build matrix from master (absolute impacts)
        mat_dest = build_matrix_from_master(
            master_df=master,
            dest_country=dest,
            top_n_rows=TOP_COUNTRIES_FOR_HEATMAP
        )

        if mat_dest.empty:
            print(f"[Skip] Empty matrix for destination {dest}")
            continue

        # Ensure columns appear as in master (then grouped)
        mat_dest = mat_dest.reindex(columns=[c for c in industries_order if c in mat_dest.columns])

        # Output paths
        out_dir = os.path.join(OUT_DIR_BASE, dest)
        os.makedirs(out_dir, exist_ok=True)
        out_png = os.path.join(out_dir, f"heatmap_dependence_shares_{dest}.png")
        out_csv_abs = os.path.join(out_dir, f"matrix_abs_impacts_{dest}.csv")
        out_csv_share = os.path.join(out_dir, f"matrix_dependence_shares_{dest}.csv")

        # Title (no group headers mentioned)
        title = (
            f"{dest} - Solar Energy Supply Chain Dependence"
            f"\nRows: shock countries; Cols: shock industries."
            f"\nShading: column shares of |negative impacts|" if SHARE_NEGATIVE_ONLY
            else f"{dest} - Solar Energy Supply Chain Dependence\nRows: shock countries; Cols: shock industries.\nShading: column shares of |all impacts|"
        )

        # Plot (shades by shares) & save
        shares_dest = plot_heatmap_dependence_shares(
            matrix=mat_dest,
            title=title,
            outpath=out_png,
            figsize=FIGSIZE
        )

        # Save the numeric matrices (rounded for readability)
        save_dataframe_rounded(mat_dest, out_csv_abs, decimals=3)      # absolute impacts used as base
        save_dataframe_rounded(shares_dest, out_csv_share, decimals=3) # shares used for shading


[Loading master] ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv
ERROR! Session/line number was not unique in database. History logging moved to new session 79
[Destinations] 49 detected: ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'GB', 'US', 'JP', 'CN', 'CA', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'CH', 'TR', 'TW', 'NO', 'ID', 'ZA', 'WA', 'WL', 'WE', 'WF', 'WM']

[Progress] 1/49 → AT
[Saved] heatmaps_from_master\AT\heatmap_dependence_shares_AT.png
[Saved] heatmaps_from_master\AT\matrix_abs_impacts_AT.csv (rounded to 3 decimals)
[Saved] heatmaps_from_master\AT\matrix_dependence_shares_AT.csv (rounded to 3 decimals)

[Progress] 2/49 → BE
[Saved] heatmaps_from_master\BE\heatmap_dependence_shares_BE.png
[Saved] heatmaps_from_master\BE\matrix_abs_impacts_BE.csv (rounded to 3 decimals)
[Saved] heatmaps_from_master\BE\matrix_dependence_shares_

In [ ]:

# -*- coding: utf-8 -*-
"""
Compute per-destination concentration metrics for solar dependence shares and save as CSV + PNG tables,
with industries grouped (Tertiary -> Secondary -> Primary) and every cell rendered on THREE LINES
WITHOUT widening columns, but with proper vertical spacing so text is readable.

Outputs:
  concentration_from_master/
    <DEST>/
      concentration_metrics_<DEST>.csv
      concentration_metrics_<DEST>.png   (paginated as _p1.png, _p2.png, ... if many rows)
    concentration_metrics__ALL_DEST.csv
    concentration_metrics__ALL_DEST.png  (paginated similarly)
"""

# ==================== CONFIG ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv"  # <-- set path
OUT_DIR_BASE = "concentration_from_master"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

DEST_LIST = None                        # None -> all destinations detected
TOP_COUNTRIES_FOR_TABLE = 49            # None -> keep all rows
SHARE_NEGATIVE_ONLY = True              # shares from negative impacts only
SAVE_ROUNDED_DECIMALS = 3

# ---------- PNG TABLE rendering (uniform widths; 3 lines per cell) ----------
PNG_ROWS_PER_PAGE = 30                  # fewer rows per page -> more room per row
UNIFORM_COL_WIDTH = 1.0                 # keep all columns same width (no widening)
CELL_H = 0.85                           # per-row height (inches) -> bigger vertical cell space
MARGIN_W = 1.0                          # left/right margins (inches)
MARGIN_H = 1.3                          # top/bottom margins (inches)

TABLE_XSCALE = 1.05                     # scale table horizontally a little
TABLE_YSCALE = 1.80                     # scale table vertically a lot (fixes "flat" PNG)

HEADER_FONTSIZE = 11
CELL_FONTSIZE = 10
HEADER_FACE_COLOR = "#F0F0F0"
ROW_ZEBRA_COLOR = "#FAFAFA"
BORDER_COLOR = "#AAAAAA"
BORDER_LW = 0.7

# Wrap width for the 'Industry' column; content is forced to 3 lines
INDUSTRY_WRAP_WIDTH = 28

# ==================== GROUPING DEFINITIONS ====================
PRIORITY_FIRST = [
    "Electrical machinery and apparatus n.e.c. (31)",
    "Fabricated metal products, except machinery and equipment (28)",
    "Machinery and equipment n.e.c. (29)",
]
METAL_PRODUCTS_BLOCK = [
    "Aluminium and aluminium products",
    "Basic iron and steel and of ferro-alloys and first products thereof",
    "Copper products",
    "Lead, zinc and tin and products thereof",
    "Other non-ferrous metal products",
    "Precious metals",
]

# ==================== HELPERS ====================
def infer_destination_list(master_columns):
    dests = []
    for c in master_columns:
        if isinstance(c, str) and c.startswith("abs_") and c.endswith("_EUR"):
            dests.append(c[len("abs_"):-len("_EUR")])
    return dests

def build_matrix_from_master(master_df, dest_country, top_n_rows=None):
    val_col = f"abs_{dest_country}_EUR"
    if val_col not in master_df.columns:
        raise KeyError(f"Destination column '{val_col}' not found in master CSV.")
    df = master_df[["Shock_Country", "Shock_Industry", val_col]].copy()
    df.rename(columns={val_col: "value"}, inplace=True)
    mat = df.pivot(index="Shock_Country", columns="Shock_Industry", values="value")
    if top_n_rows is not None and top_n_rows > 0:
        mag = mat.abs().sum(axis=1).sort_values(ascending=False)
        keep_rows = list(mag.head(top_n_rows).index)
        mat = mat.loc[keep_rows, :]
    return mat

def compute_shares_matrix(abs_matrix: pd.DataFrame, negative_only: bool = True) -> pd.DataFrame:
    m = abs_matrix.copy()
    m = m.mask(np.isfinite(m) & (np.abs(m) < 1e-12), 0.0)
    if negative_only:
        base = m.where(m < 0.0)
        numer = base.abs()
    else:
        numer = m.abs()
    denom = numer.sum(axis=0)
    shares = numer.divide(denom.where(denom != 0), axis=1)
    return shares

def _concentration_metrics_for_column(col_shares: pd.Series) -> dict:
    s = col_shares.dropna()
    if s.empty or s.sum() == 0:
        return {
            "N": 0, "HHI": np.nan, "HHI_norm": np.nan, "Eff_N_hhi": np.nan,
            "CR1": np.nan, "CR3": np.nan, "CR5": np.nan,
            "Entropy": np.nan, "Entropy_norm": np.nan, "Eff_N_entropy": np.nan,
        }
    p = s / s.sum()
    N = len(p)
    hhi = float((p ** 2).sum())
    eff_N_hhi = (1.0 / hhi) if hhi > 0 else np.nan
    hhi_min = 1.0 / N
    hhi_norm = ((hhi - hhi_min) / (1.0 - hhi_min)) if N > 1 else np.nan
    cr1 = float(p.max())
    cr3 = float(p.nlargest(min(3, N)).sum())
    cr5 = float(p.nlargest(min(5, N)).sum())
    entropy = float(- (p * np.log(p)).sum())
    entropy_norm = (entropy / np.log(N)) if N > 1 else np.nan
    eff_N_entropy = float(np.exp(entropy))
    return {
        "N": N, "HHI": hhi, "HHI_norm": hhi_norm, "Eff_N_hhi": eff_N_hhi,
        "CR1": cr1, "CR3": cr3, "CR5": cr5,
        "Entropy": entropy, "Entropy_norm": entropy_norm, "Eff_N_entropy": eff_N_entropy,
    }

def compute_concentration_table(shares_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ind in shares_df.columns:
        m = _concentration_metrics_for_column(shares_df[ind])
        m["Industry"] = ind
        rows.append(m)
    return pd.DataFrame(rows).set_index("Industry")

# ----- Industry grouping -----
def compute_industry_group_order(present_inds):
    present_set = set(present_inds)
    priority = [c for c in PRIORITY_FIRST if c in present_set]
    metals   = [c for c in METAL_PRODUCTS_BLOCK if c in present_set]
    rest     = sorted([c for c in present_inds if c not in set(PRIORITY_FIRST + METAL_PRODUCTS_BLOCK)])
    ordered = priority + metals + rest
    blocks = []
    stage_map = {}
    start = 0
    if len(priority) > 0:
        blocks.append(("Tertiary", start, start + len(priority) - 1))
        for x in priority: stage_map[x] = "Tertiary"
        start += len(priority)
    if len(metals) > 0:
        blocks.append(("Secondary", start, start + len(metals) - 1))
        for x in metals: stage_map[x] = "Secondary"
        start += len(metals)
    if len(rest) > 0:
        blocks.append(("Primary", start, start + len(rest) - 1))
        for x in rest: stage_map[x] = "Primary"
    return ordered, blocks, stage_map

def save_dataframe_rounded(df: pd.DataFrame, path: str, decimals: int = 3):
    df_round = df.copy()
    with np.errstate(invalid='ignore'):
        df_round = df_round.round(decimals)
    df_round.to_csv(path)
    print(f"[Saved] {path} (rounded to {decimals} decimals)")

# ==================== 3-LINE CELL RENDERING HELPERS ====================
def _wrap_to_three_lines(text: str, width: int) -> str:
    lines = textwrap.wrap(str(text), width=width) or [""]
    if len(lines) > 3:
        lines = lines[:3]
        lines[-1] = lines[-1].rstrip() + "…"
    while len(lines) < 3:
        lines.append("")
    return "\n".join(lines)

def _center_number_three_lines(val, decimals: int) -> str:
    if pd.isna(val):
        return "\n\n"
    return f"\n{val:.{decimals}f}\n"

def _center_text_three_lines(s: str) -> str:
    s = "" if s is None else str(s)
    return f"\n{s}\n"

# ==================== PNG TABLE RENDERING ====================
def _df_to_table_page(df: pd.DataFrame, title: str, out_png: str,
                      uniform_col_width=UNIFORM_COL_WIDTH,
                      header_fs=HEADER_FONTSIZE, cell_fs=CELL_FONTSIZE,
                      cell_h=CELL_H, margin_w=MARGIN_W, margin_h=MARGIN_H,
                      zebra_color=ROW_ZEBRA_COLOR, header_color=HEADER_FACE_COLOR,
                      decimals=SAVE_ROUNDED_DECIMALS, industry_wrap_width=INDUSTRY_WRAP_WIDTH,
                      table_xscale=TABLE_XSCALE, table_yscale=TABLE_YSCALE):
    """
    Render a single DataFrame page to PNG with Matplotlib's table,
    enforcing EXACTLY 3 lines per cell; NO widening of columns; scale vertically.
    """
    n_rows, n_cols = df.shape

    # Uniform column widths
    col_widths = [uniform_col_width] * n_cols

    # Figure dimensions
    fig_w = margin_w * 2 + sum(col_widths)
    fig_h = margin_h * 2 + cell_h * (n_rows + 1)  # +1 for header

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    # Prepare display DF with 3-line content
    display_df = df.copy()
    for c in display_df.columns:
        if c == "Industry":
            display_df[c] = display_df[c].apply(lambda s: _wrap_to_three_lines(s, industry_wrap_width))
        elif pd.api.types.is_numeric_dtype(display_df[c]):
            display_df[c] = display_df[c].apply(lambda x: _center_number_three_lines(x, decimals))
        else:
            display_df[c] = display_df[c].apply(_center_text_three_lines)

    cell_text = display_df.values.tolist()
    col_labels = list(df.columns)

    # Create table
    tbl = ax.table(cellText=cell_text, colLabels=col_labels, colWidths=col_widths,
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(cell_fs)

    # >>> Critical: scale the table so rows are taller <<<
    tbl.scale(table_xscale, table_yscale)

    # Header style
    for j in range(n_cols):
        c = tbl.get_celld()[(0, j)]
        c.set_facecolor(header_color)
        c.set_edgecolor(BORDER_COLOR)
        c.set_linewidth(BORDER_LW)
        c.set_text_props(weight='bold', fontsize=header_fs)

    # Body style
    for i in range(1, n_rows + 1):
        for j in range(n_cols):
            c = tbl.get_celld()[(i, j)]
            if i % 2 == 0:
                c.set_facecolor(zebra_color)
            c.set_edgecolor(BORDER_COLOR)
            c.set_linewidth(BORDER_LW)
            if df.columns[j] == "Industry":
                c.set_text_props(ha='left')
            else:
                c.set_text_props(ha='center')

    # Title
    ax.set_title(title, fontsize=header_fs + 2, pad=8)

    # Save
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[Saved] {out_png}")

def save_table_png(df: pd.DataFrame, base_png_path: str, title: str, rows_per_page: int = PNG_ROWS_PER_PAGE):
    preferred_order = ["Industry", "Stage", "N", "HHI", "HHI_norm", "Eff_N_hhi",
                       "CR1", "CR3", "CR5", "Entropy", "Entropy_norm", "Eff_N_entropy"]
    cols = [c for c in preferred_order if c in df.columns] + [c for c in df.columns if c not in preferred_order]
    df = df[cols]

    n = len(df)
    if rows_per_page is None or n <= rows_per_page:
        out_png = base_png_path if base_png_path.endswith(".png") else base_png_path + ".png"
        _df_to_table_page(df, title, out_png)
        return [out_png]
    else:
        pages = math.ceil(n / rows_per_page)
        pngs = []
        for p in range(pages):
            start = p * rows_per_page
            end = min((p + 1) * rows_per_page, n)
            df_page = df.iloc[start:end, :]
            out_png = base_png_path.replace(".png", "") + f"_p{p+1}.png"
            _df_to_table_page(df_page, f"{title} (page {p+1}/{pages})", out_png)
            pngs.append(out_png)
        return pngs

# ==================== MAIN ====================
if __name__ == "__main__":
    if not os.path.exists(MASTER_CSV):
        raise FileNotFoundError(f"MASTER_CSV not found: {MASTER_CSV}")
    print(f"[Loading master] {MASTER_CSV}")
    master = pd.read_csv(MASTER_CSV)

    detected_destinations = infer_destination_list(master.columns)
    if DEST_LIST is None:
        DEST_LIST = detected_destinations
    else:
        DEST_LIST = [d for d in DEST_LIST if f"abs_{d}_EUR" in master.columns]
    if not DEST_LIST:
        raise ValueError("No destination columns found in master CSV (abs_<CC>_EUR).")
    print(f"[Destinations] {len(DEST_LIST)} detected: {DEST_LIST}")

    for req in ("Shock_Industry", "Shock_Country"):
        if req not in master.columns:
            raise ValueError(f"Master CSV missing column '{req}'.")

    industries_order = list(pd.Index(master["Shock_Industry"]).unique())

    all_records = []
    total = len(DEST_LIST)
    for idx, dest in enumerate(DEST_LIST, start=1):
        print(f"\n[Progress] {idx}/{total} → {dest}")

        mat_abs = build_matrix_from_master(master_df=master, dest_country=dest, top_n_rows=TOP_COUNTRIES_FOR_TABLE)
        if mat_abs.empty:
            print(f"[Skip] Empty matrix for destination {dest}")
            continue

        mat_abs = mat_abs.reindex(columns=[c for c in industries_order if c in mat_abs.columns])
        shares = compute_shares_matrix(mat_abs, negative_only=SHARE_NEGATIVE_ONLY)
        conc_df = compute_concentration_table(shares)  # index=Industry

        present_inds = list(conc_df.index)
        ordered_inds, row_blocks, stage_map = compute_industry_group_order(present_inds)
        conc_df = conc_df.reindex(index=ordered_inds)
        conc_df.insert(0, "Stage", conc_df.index.map(stage_map))

        out_dir = os.path.join(OUT_DIR_BASE, dest)
        os.makedirs(out_dir, exist_ok=True)
        out_csv_conc = os.path.join(out_dir, f"concentration_metrics_{dest}.csv")
        out_png_base = os.path.join(out_dir, f"concentration_metrics_{dest}.png")

        save_dataframe_rounded(conc_df, out_csv_conc, decimals=SAVE_ROUNDED_DECIMALS)

        _ = save_table_png(
            conc_df.reset_index().rename(columns={"index": "Industry"}),
            base_png_path=out_png_base,
            title=f"{dest} — Concentration metrics per shock industry" + (
                " (shares of |negative impacts|)" if SHARE_NEGATIVE_ONLY else " (shares of |all impacts|)"
            ),
            rows_per_page=PNG_ROWS_PER_PAGE
        )

        df_tmp = conc_df.reset_index().rename(columns={"index": "Industry"})
        df_tmp.insert(0, "Destination", dest)
        all_records.append(df_tmp)

    if all_records:
        combined = pd.concat(all_records, ignore_index=True)
        out_combined_csv = os.path.join(OUT_DIR_BASE, "concentration_metrics__ALL_DEST.csv")
        out_combined_png = os.path.join(OUT_DIR_BASE, "concentration_metrics__ALL_DEST.png")
        save_dataframe_rounded(combined, out_combined_csv, decimals=SAVE_ROUNDED_DECIMALS)
        _ = save_table_png(
            combined,
            base_png_path=out_combined_png,
            title="All destinations — Concentration metrics per shock industry",
            rows_per_page=PNG_ROWS_PER_PAGE
        )

    print("\n[Done] Grouped concentration tables (CSV + PNG, 3 lines per cell, vertically scaled) generated.")

In [14]:
# -*- coding: utf-8 -*-
"""
Compute per-destination concentration metrics for solar dependence shares and save as CSV + PNG tables,
with industries grouped (Tertiary -> Secondary -> Primary) and every cell rendered on THREE LINES
WITHOUT widening columns, but with proper vertical spacing so text is readable.

NEW: Adds 'SelfReliance' = domestic share per (Destination x Industry), consistent with the shares matrix:
      SelfReliance[DEST, ind] = shares.loc[DEST, ind]  (0..1)

Outputs:
  concentration_from_master/
    <DEST>/
      concentration_metrics_<DEST>.csv
      concentration_metrics_<DEST>.png   (paginated as _p1.png, _p2.png, ... if many rows)
    concentration_metrics__ALL_DEST.csv
    concentration_metrics__ALL_DEST.png  (paginated similarly)
"""

# ==================== IMPORTS ====================
import os
import math
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==================== CONFIG ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv"  # <-- set path
OUT_DIR_BASE = "concentration_from_master"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

DEST_LIST = None                        # None -> all destinations detected
TOP_COUNTRIES_FOR_TABLE = 49            # None -> keep all rows; if set, DEST row is force-kept
SHARE_NEGATIVE_ONLY = True              # shares from negative impacts only
SAVE_ROUNDED_DECIMALS = 3

# ---------- PNG TABLE rendering (uniform widths; 3 lines per cell) ----------
PNG_ROWS_PER_PAGE = 30                  # fewer rows per page -> more room per row
UNIFORM_COL_WIDTH = 1.0                 # keep all columns same width (no widening)
CELL_H = 0.85                           # per-row height (inches) -> bigger vertical cell space
MARGIN_W = 1.0                          # left/right margins (inches)
MARGIN_H = 1.3                          # top/bottom margins (inches)

TABLE_XSCALE = 1.05                     # scale table horizontally a little
TABLE_YSCALE = 1.80                     # scale table vertically a lot (fixes "flat" PNG)

HEADER_FONTSIZE = 11
CELL_FONTSIZE = 10
HEADER_FACE_COLOR = "#F0F0F0"
ROW_ZEBRA_COLOR = "#FAFAFA"
BORDER_COLOR = "#AAAAAA"
BORDER_LW = 0.7

# Wrap width for the 'Industry' column; content is forced to 3 lines
INDUSTRY_WRAP_WIDTH = 28

# ==================== GROUPING DEFINITIONS ====================
PRIORITY_FIRST = [
    "Electrical machinery and apparatus n.e.c. (31)",
    "Fabricated metal products, except machinery and equipment (28)",
    "Machinery and equipment n.e.c. (29)",
]
METAL_PRODUCTS_BLOCK = [
    "Aluminium and aluminium products",
    "Basic iron and steel and of ferro-alloys and first products thereof",
    "Copper products",
    "Lead, zinc and tin and products thereof",
    "Other non-ferrous metal products",
    "Precious metals",
]

# ==================== HELPERS ====================
def infer_destination_list(master_columns):
    dests = []
    for c in master_columns:
        if isinstance(c, str) and c.startswith("abs_") and c.endswith("_EUR"):
            dests.append(c[len("abs_"):-len("_EUR")])
    return dests


def build_matrix_from_master(master_df, dest_country, top_n_rows=None):
    """
    Build matrix of absolute impacts with rows = Shock_Country, cols = Shock_Industry for a given destination.
    If top_n_rows is set, keep the top rows by total |impact| and ALWAYS keep dest_country row if present.
    """
    val_col = f"abs_{dest_country}_EUR"
    if val_col not in master_df.columns:
        raise KeyError(f"Destination column '{val_col}' not found in master CSV.")
    df = master_df[["Shock_Country", "Shock_Industry", val_col]].copy()
    df.rename(columns={val_col: "value"}, inplace=True)
    mat = df.pivot(index="Shock_Country", columns="Shock_Industry", values="value")

    if top_n_rows is not None and top_n_rows > 0 and len(mat) > top_n_rows:
        # rank by magnitude across industries
        mag = mat.abs().sum(axis=1).sort_values(ascending=False)
        keep_rows = list(mag.head(top_n_rows).index)
        # force-keep destination row if it exists in the matrix but was not selected
        if dest_country in mat.index and dest_country not in keep_rows:
            keep_rows.append(dest_country)
        mat = mat.loc[keep_rows, :]

    return mat


def compute_shares_matrix(abs_matrix: pd.DataFrame, negative_only: bool = True) -> pd.DataFrame:
    """
    Column-normalize absolute values (or only negatives) to get shares that sum to 1 per industry column.
    """
    m = abs_matrix.copy()
    m = m.mask(np.isfinite(m) & (np.abs(m) < 1e-12), 0.0)
    if negative_only:
        base = m.where(m < 0.0)
        numer = base.abs()
    else:
        numer = m.abs()
    denom = numer.sum(axis=0)
    shares = numer.divide(denom.where(denom != 0), axis=1)
    return shares


def _concentration_metrics_for_column(col_shares: pd.Series) -> dict:
    s = col_shares.dropna()
    if s.empty or s.sum() == 0:
        return {
            "N": 0, "HHI": np.nan, "HHI_norm": np.nan, "Eff_N_hhi": np.nan,
            "CR1": np.nan, "CR3": np.nan, "CR5": np.nan,
            "Entropy": np.nan, "Entropy_norm": np.nan, "Eff_N_entropy": np.nan,
        }
    p = s / s.sum()
    N = len(p)
    hhi = float((p ** 2).sum())
    eff_N_hhi = (1.0 / hhi) if hhi > 0 else np.nan
    hhi_min = 1.0 / N
    hhi_norm = ((hhi - hhi_min) / (1.0 - hhi_min)) if N > 1 else np.nan
    cr1 = float(p.max())
    cr3 = float(p.nlargest(min(3, N)).sum())
    cr5 = float(p.nlargest(min(5, N)).sum())
    entropy = float(- (p * np.log(p)).sum())
    entropy_norm = (entropy / np.log(N)) if N > 1 else np.nan
    eff_N_entropy = float(np.exp(entropy))
    return {
        "N": N, "HHI": hhi, "HHI_norm": hhi_norm, "Eff_N_hhi": eff_N_hhi,
        "CR1": cr1, "CR3": cr3, "CR5": cr5,
        "Entropy": entropy, "Entropy_norm": entropy_norm, "Eff_N_entropy": eff_N_entropy,
    }


def compute_concentration_table(shares_df: pd.DataFrame, dest_country: str) -> pd.DataFrame:
    """
    For each industry (column), compute standard concentration metrics + SelfReliance
    (domestic share for dest_country).
    """
    rows = []
    for ind in shares_df.columns:
        col = shares_df[ind]
        m = _concentration_metrics_for_column(col)

        # -------- Self‑reliance (domestic share) --------
        if dest_country in shares_df.index:
            # If present, get the share; if NaN, treat as 0 (no contribution)
            val = col.loc[dest_country]
            m["SelfReliance"] = float(val) if pd.notna(val) else 0.0
        else:
            # Not in the shares index: treat as 0 contribution
            m["SelfReliance"] = 0.0

        m["Industry"] = ind
        rows.append(m)

    return pd.DataFrame(rows).set_index("Industry")


# ----- Industry grouping -----
def compute_industry_group_order(present_inds):
    present_set = set(present_inds)
    priority = [c for c in PRIORITY_FIRST if c in present_set]
    metals   = [c for c in METAL_PRODUCTS_BLOCK if c in present_set]
    rest     = sorted([c for c in present_inds if c not in set(PRIORITY_FIRST + METAL_PRODUCTS_BLOCK)])
    ordered = priority + metals + rest
    blocks = []
    stage_map = {}
    start = 0
    if len(priority) > 0:
        blocks.append(("Tertiary", start, start + len(priority) - 1))
        for x in priority: stage_map[x] = "Tertiary"
        start += len(priority)
    if len(metals) > 0:
        blocks.append(("Secondary", start, start + len(metals) - 1))
        for x in metals: stage_map[x] = "Secondary"
        start += len(metals)
    if len(rest) > 0:
        blocks.append(("Primary", start, start + len(rest) - 1))
        for x in rest: stage_map[x] = "Primary"
    return ordered, blocks, stage_map


def save_dataframe_rounded(df: pd.DataFrame, path: str, decimals: int = 3):
    df_round = df.copy()
    with np.errstate(invalid='ignore'):
        df_round = df_round.round(decimals)
    df_round.to_csv(path, index=True)
    print(f"[Saved] {path} (rounded to {decimals} decimals)")


# ==================== 3-LINE CELL RENDERING HELPERS ====================
def _wrap_to_three_lines(text: str, width: int) -> str:
    lines = textwrap.wrap(str(text), width=width) or [""]
    if len(lines) > 3:
        lines = lines[:3]
        lines[-1] = lines[-1].rstrip() + "…"
    while len(lines) < 3:
        lines.append("")
    return "\n".join(lines)


def _center_number_three_lines(val, decimals: int) -> str:
    if pd.isna(val):
        return "\n\n"
    return f"\n{val:.{decimals}f}\n"


def _center_text_three_lines(s: str) -> str:
    s = "" if s is None else str(s)
    return f"\n{s}\n"


# ==================== PNG TABLE RENDERING ====================
def _df_to_table_page(df: pd.DataFrame, title: str, out_png: str,
                      uniform_col_width=UNIFORM_COL_WIDTH,
                      header_fs=HEADER_FONTSIZE, cell_fs=CELL_FONTSIZE,
                      cell_h=CELL_H, margin_w=MARGIN_W, margin_h=MARGIN_H,
                      zebra_color=ROW_ZEBRA_COLOR, header_color=HEADER_FACE_COLOR,
                      decimals=SAVE_ROUNDED_DECIMALS, industry_wrap_width=INDUSTRY_WRAP_WIDTH,
                      table_xscale=TABLE_XSCALE, table_yscale=TABLE_YSCALE):
    """
    Render a single DataFrame page to PNG with Matplotlib's table,
    enforcing EXACTLY 3 lines per cell; NO widening of columns; scale vertically.
    """
    n_rows, n_cols = df.shape

    # Uniform column widths
    col_widths = [uniform_col_width] * n_cols

    # Figure dimensions
    fig_w = margin_w * 2 + sum(col_widths)
    fig_h = margin_h * 2 + cell_h * (n_rows + 1)  # +1 for header

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    # Prepare display DF with 3-line content
    display_df = df.copy()
    for c in display_df.columns:
        if c == "Industry":
            display_df[c] = display_df[c].apply(lambda s: _wrap_to_three_lines(s, industry_wrap_width))
        elif pd.api.types.is_numeric_dtype(display_df[c]):
            display_df[c] = display_df[c].apply(lambda x: _center_number_three_lines(x, decimals))
        else:
            display_df[c] = display_df[c].apply(_center_text_three_lines)

    cell_text = display_df.values.tolist()
    col_labels = list(df.columns)

    # Create table
    tbl = ax.table(cellText=cell_text, colLabels=col_labels, colWidths=col_widths,
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(cell_fs)

    # >>> Critical: scale the table so rows are taller <<<
    tbl.scale(table_xscale, table_yscale)

    # Header style
    for j in range(n_cols):
        c = tbl.get_celld()[(0, j)]
        c.set_facecolor(header_color)
        c.set_edgecolor(BORDER_COLOR)
        c.set_linewidth(BORDER_LW)
        c.set_text_props(weight='bold', fontsize=header_fs)

    # Body style
    for i in range(1, n_rows + 1):
        for j in range(n_cols):
            c = tbl.get_celld()[(i, j)]
            if i % 2 == 0:
                c.set_facecolor(zebra_color)
            c.set_edgecolor(BORDER_COLOR)
            c.set_linewidth(BORDER_LW)
            if df.columns[j] == "Industry":
                c.set_text_props(ha='left')
            else:
                c.set_text_props(ha='center')

    # Title
    ax.set_title(title, fontsize=header_fs + 2, pad=8)

    # Save
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[Saved] {out_png}")


def save_table_png(df: pd.DataFrame, base_png_path: str, title: str, rows_per_page: int = PNG_ROWS_PER_PAGE):
    preferred_order = [
        "Industry", "Stage", "SelfReliance",
        "N", "HHI", "HHI_norm", "Eff_N_hhi",
        "CR1", "CR3", "CR5",
        "Entropy", "Entropy_norm", "Eff_N_entropy"
    ]
    cols = [c for c in preferred_order if c in df.columns] + [c for c in df.columns if c not in preferred_order]
    df = df[cols]

    n = len(df)
    if rows_per_page is None or n <= rows_per_page:
        out_png = base_png_path if base_png_path.endswith(".png") else base_png_path + ".png"
        _df_to_table_page(df, title, out_png)
        return [out_png]
    else:
        pages = math.ceil(n / rows_per_page)
        pngs = []
        for p in range(pages):
            start = p * rows_per_page
            end = min((p + 1) * rows_per_page, n)
            df_page = df.iloc[start:end, :]
            out_png = base_png_path.replace(".png", "") + f"_p{p+1}.png"
            _df_to_table_page(df_page, f"{title} (page {p+1}/{pages})", out_png)
            pngs.append(out_png)
        return pngs


# ==================== MAIN ====================
if __name__ == "__main__":
    if not os.path.exists(MASTER_CSV):
        raise FileNotFoundError(f"MASTER_CSV not found: {MASTER_CSV}")
    print(f"[Loading master] {MASTER_CSV}")
    master = pd.read_csv(MASTER_CSV)

    detected_destinations = infer_destination_list(master.columns)
    if DEST_LIST is None:
        DEST_LIST = detected_destinations
    else:
        DEST_LIST = [d for d in DEST_LIST if f"abs_{d}_EUR" in master.columns]
    if not DEST_LIST:
        raise ValueError("No destination columns found in master CSV (abs_<CC>_EUR).")
    print(f"[Destinations] {len(DEST_LIST)} detected: {DEST_LIST}")

    for req in ("Shock_Industry", "Shock_Country"):
        if req not in master.columns:
            raise ValueError(f"Master CSV missing column '{req}'.")

    industries_order = list(pd.Index(master["Shock_Industry"]).unique())

    all_records = []
    total = len(DEST_LIST)
    for idx, dest in enumerate(DEST_LIST, start=1):
        print(f"\n[Progress] {idx}/{total} → {dest}")

        mat_abs = build_matrix_from_master(master_df=master, dest_country=dest, top_n_rows=TOP_COUNTRIES_FOR_TABLE)
        if mat_abs.empty:
            print(f"[Skip] Empty matrix for destination {dest}")
            continue

        # Column order: respect discovery order filtered to those present
        mat_abs = mat_abs.reindex(columns=[c for c in industries_order if c in mat_abs.columns])
        shares = compute_shares_matrix(mat_abs, negative_only=SHARE_NEGATIVE_ONLY)

        # Concentration + SelfReliance
        conc_df = compute_concentration_table(shares, dest_country=dest)  # index=Industry

        # Grouping and Stage labels
        present_inds = list(conc_df.index)
        ordered_inds, row_blocks, stage_map = compute_industry_group_order(present_inds)
        conc_df = conc_df.reindex(index=ordered_inds)
        conc_df.insert(0, "Stage", conc_df.index.map(stage_map))

        # Out paths
        out_dir = os.path.join(OUT_DIR_BASE, dest)
        os.makedirs(out_dir, exist_ok=True)
        out_csv_conc = os.path.join(out_dir, f"concentration_metrics_{dest}.csv")
        out_png_base = os.path.join(out_dir, f"concentration_metrics_{dest}.png")

        # CSV
        save_dataframe_rounded(conc_df, out_csv_conc, decimals=SAVE_ROUNDED_DECIMALS)

        # PNG (3-line cells)
        _ = save_table_png(
            conc_df.reset_index().rename(columns={"index": "Industry"}),
            base_png_path=out_png_base,
            title=f"{dest} — Concentration metrics per shock industry" + (
                " (shares of |negative impacts|)" if SHARE_NEGATIVE_ONLY else " (shares of |all impacts|)"
            ),
            rows_per_page=PNG_ROWS_PER_PAGE
        )

        # For combined table
        df_tmp = conc_df.reset_index().rename(columns={"index": "Industry"})
        df_tmp.insert(0, "Destination", dest)
        all_records.append(df_tmp)

    if all_records:
        combined = pd.concat(all_records, ignore_index=True)
        out_combined_csv = os.path.join(OUT_DIR_BASE, "concentration_metrics__ALL_DEST.csv")
        out_combined_png = os.path.join(OUT_DIR_BASE, "concentration_metrics__ALL_DEST.png")
        save_dataframe_rounded(combined, out_combined_csv, decimals=SAVE_ROUNDED_DECIMALS)
        _ = save_table_png(
            combined,
            base_png_path=out_combined_png,
            title="All destinations — Concentration metrics per shock industry",
            rows_per_page=PNG_ROWS_PER_PAGE
        )

    print("\n[Done] Grouped concentration tables (CSV + PNG, 3 lines per cell, vertically scaled) generated, with SelfReliance.")

[Loading master] ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20260420_1040.csv
[Destinations] 49 detected: ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'GB', 'US', 'JP', 'CN', 'CA', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'CH', 'TR', 'TW', 'NO', 'ID', 'ZA', 'WA', 'WL', 'WE', 'WF', 'WM']

[Progress] 1/49 → AT
[Saved] concentration_from_master\AT\concentration_metrics_AT.csv (rounded to 3 decimals)


C:\Users\Dyde-nairn\AppData\Local\Temp\ipykernel_20252\1053751061.py:299: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


[Saved] concentration_from_master\AT\concentration_metrics_AT.png

[Progress] 2/49 → BE
[Saved] concentration_from_master\BE\concentration_metrics_BE.csv (rounded to 3 decimals)


C:\Users\Dyde-nairn\AppData\Local\Temp\ipykernel_20252\1053751061.py:299: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


[Saved] concentration_from_master\BE\concentration_metrics_BE.png

[Progress] 3/49 → BG
[Saved] concentration_from_master\BG\concentration_metrics_BG.csv (rounded to 3 decimals)


...

[Saved] concentration_from_master\WM\concentration_metrics_WM.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST.csv (rounded to 3 decimals)


C:\Users\Dyde-nairn\AppData\Local\Temp\ipykernel_20252\1053751061.py:299: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p1.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p2.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p3.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p4.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p5.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p6.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p7.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p8.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p9.png
[Saved] concentration_from_master\concentration_metrics__ALL_DEST_p10.png


KeyboardInterrupt: 

KeyboardInterrupt: 

In [3]:
# -*- coding: utf-8 -*-
"""
China sector-specific upstream dependence of solar output
(end-use normalized)

Each cell shows:
  Share (%) of a destination country's total upstream exposure
  in a given sector that is attributable to China.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

# ==================== PATHS ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20251222_1207.csv"

OUT_DIR = "china_sector_dependence"
OUT_PNG = os.path.join(OUT_DIR, "china_sector_dependence_solar.png")
os.makedirs(OUT_DIR, exist_ok=True)

# ==================== SETTINGS ====================
FIGSIZE = (20, 12)
WRAP_WIDTH = 28
TOP_N_DESTINATIONS = 30     # set None to keep all
USE_NEGATIVE_ONLY = True    # recommended

# ==================== HELPERS ====================
def infer_destinations(columns):
    return [
        c[len("abs_"):-len("_EUR")]
        for c in columns
        if c.startswith("abs_") and c.endswith("_EUR")
    ]

def wrap_label(s, width=28):
    return "\n".join(textwrap.wrap(str(s), width=width))

# ==================== LOAD DATA ====================
master = pd.read_csv(MASTER_CSV)

destinations = infer_destinations(master.columns)
sectors = master["Shock_Industry"].unique()

# ==================== CORE COMPUTATION ====================
# Result: rows = destination countries, columns = sectors
china_share = pd.DataFrame(index=destinations, columns=sectors, dtype=float)

for dest in destinations:
    dest_col = f"abs_{dest}_EUR"
    df_d = master[["Shock_Country", "Shock_Industry", dest_col]].copy()
    df_d.rename(columns={dest_col: "impact"}, inplace=True)

    if USE_NEGATIVE_ONLY:
        df_d = df_d[df_d["impact"] < 0]
        df_d["impact"] = df_d["impact"].abs()
    else:
        df_d["impact"] = df_d["impact"].abs()

    # Sum over shock countries, sector by sector
    sector_totals = df_d.groupby("Shock_Industry")["impact"].sum()

    # China contribution
    df_cn = df_d[df_d["Shock_Country"] == "CN"]
    china_sector = df_cn.set_index("Shock_Industry")["impact"]

    # Share
    china_share.loc[dest, sector_totals.index] = (
        china_sector / sector_totals
    )

# Convert to %
china_share *= 100

# ==================== OPTIONAL: TOP DESTINATIONS ====================
if TOP_N_DESTINATIONS is not None:
    mag = china_share.sum(axis=1).sort_values(ascending=False)
    china_share = china_share.loc[mag.head(TOP_N_DESTINATIONS).index]

# ==================== PLOT ====================
fig, ax = plt.subplots(figsize=FIGSIZE)

cmap = plt.get_cmap("Reds").copy()
cmap.set_bad("white")

im = ax.imshow(china_share.values, aspect="auto", cmap=cmap, vmin=0, vmax=100)

ax.set_xticks(np.arange(china_share.shape[1]))
ax.set_xticklabels(
    [wrap_label(c, WRAP_WIDTH) for c in china_share.columns],
    rotation=0,
    ha="center",
    fontsize=9
)

ax.set_yticks(np.arange(china_share.shape[0]))
ax.set_yticklabels(china_share.index, fontsize=10)

# Gridlines
for j in range(china_share.shape[1] - 1):
    ax.axvline(j + 0.5, color="grey", lw=0.6, alpha=0.3)
for i in range(china_share.shape[0] - 1):
    ax.axhline(i + 0.5, color="grey", lw=0.6, alpha=0.3)

ax.set_title(
    "China’s sector-specific upstream share in national solar output exposure\n"
    "Rows: destination countries · Columns: upstream sectors\n"
    "Values: % of total upstream exposure attributable to China",
    fontsize=14,
    pad=14
)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("China share of upstream exposure (%)")

plt.subplots_adjust(bottom=0.30)
plt.savefig(OUT_PNG, bbox_inches="tight", dpi=300)
plt.close()

print(f"[Saved] {OUT_PNG}")

[Saved] china_sector_dependence\china_sector_dependence_solar.png


In [8]:
# -*- coding: utf-8 -*-
"""
China sector-specific upstream dependence of solar output (end-use normalized)

Rows:
  Top GDP destination countries (exogenous selection)

Columns (left → right):
  1. Other / undefined sectors
  2. Metal & material processing sectors
  3. Key downstream manufacturing sectors

Each cell shows:
  Share (%) of a destination country's total upstream exposure
  in a given sector that is attributable to China.

Annotations:
  Only China-dominant cases are annotated (share >= threshold).
"""

# ==================== IMPORTS ====================
import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==================== PATHS ====================
MASTER_CSV = "ghosh_shocks_to_solar/master__solar_impacts_allcountries__1pct__20251222_1207.csv"

OUT_DIR = "china_sector_dependence"
OUT_PNG = os.path.join(OUT_DIR, "china_sector_dependence_solar_topGDP.png")
os.makedirs(OUT_DIR, exist_ok=True)

# ==================== SETTINGS ====================
FIGSIZE = (20, 12)
WRAP_WIDTH = 28
USE_NEGATIVE_ONLY = True

# --- Top GDP countries (ID and TR removed) ---
TOP_GDP_COUNTRIES = [
    "US", "CN", "JP", "DE", "IN", "GB", "FR", "IT", "BR", "CA",
    "RU", "KR", "AU", "ES", "MX", "NL", "SA", "CH"
]

# --- Annotation settings (China-dominant only) ---
ANNOTATE = True
CHINA_DOMINANCE_THRESHOLD = 30.0
ANNOTATION_DECIMALS = 1
TEXT_COLOR_DARK = "white"
TEXT_COLOR_LIGHT = "black"

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

# ==================== COLUMN GROUPS ====================
PRIORITY_FIRST = [
    "Electrical machinery and apparatus n.e.c. (31)",
    "Fabricated metal products, except machinery and equipment (28)",
    "Machinery and equipment n.e.c. (29)",
]

METAL_PRODUCTS_BLOCK = [
    "Aluminium and aluminium products",
    "Basic iron and steel and of ferro-alloys and first products thereof",
    "Copper products",
    "Lead, zinc and tin and products thereof",
    "Other non-ferrous metal products",
    "Precious metals",
]

# ==================== HELPERS ====================
def infer_destinations(columns):
    return [
        c[len("abs_"):-len("_EUR")]
        for c in columns
        if isinstance(c, str) and c.startswith("abs_") and c.endswith("_EUR")
    ]

def wrap_label(s, width=28):
    return "\n".join(textwrap.wrap(str(s), width=width))

def compute_ordered_columns_and_blocks(present_cols):
    """
    Desired column order (left → right):
      1. REST (undefined sectors)
      2. METAL_PRODUCTS_BLOCK
      3. PRIORITY_FIRST
    """
    present_set = set(present_cols)

    priority = [c for c in PRIORITY_FIRST if c in present_set]
    metals   = [c for c in METAL_PRODUCTS_BLOCK if c in present_set]
    rest     = sorted([
        c for c in present_cols
        if c not in set(PRIORITY_FIRST + METAL_PRODUCTS_BLOCK)
    ])

    ordered_cols = rest + metals + priority
    return ordered_cols

# ==================== LOAD DATA ====================
print("[Loading master CSV]")
master = pd.read_csv(MASTER_CSV)

required_cols = {"Shock_Country", "Shock_Industry"}
if not required_cols.issubset(master.columns):
    raise ValueError(f"Master CSV missing required columns: {required_cols}")

destinations = infer_destinations(master.columns)
sectors = master["Shock_Industry"].unique()

# ==================== CORE COMPUTATION ====================
china_share = pd.DataFrame(index=destinations, columns=sectors, dtype=float)

for dest in destinations:
    col = f"abs_{dest}_EUR"
    if col not in master.columns:
        continue

    df = master[["Shock_Country", "Shock_Industry", col]].copy()
    df.rename(columns={col: "impact"}, inplace=True)

    if USE_NEGATIVE_ONLY:
        df = df[df["impact"] < 0]
        df["impact"] = df["impact"].abs()
    else:
        df["impact"] = df["impact"].abs()

    totals = df.groupby("Shock_Industry")["impact"].sum()
    china = df[df["Shock_Country"] == "CN"].set_index("Shock_Industry")["impact"]

    china_share.loc[dest, totals.index] = china / totals

china_share *= 100

# ==================== KEEP TOP GDP COUNTRIES ====================
china_share = china_share.loc[
    [c for c in TOP_GDP_COUNTRIES if c in china_share.index]
]

# ==================== ORDER COLUMNS ====================
ordered_cols = compute_ordered_columns_and_blocks(list(china_share.columns))
china_share = china_share.reindex(columns=ordered_cols)

# ==================== PLOT ====================
fig, ax = plt.subplots(figsize=FIGSIZE)

cmap = plt.get_cmap("Reds").copy()
cmap.set_bad("white")

im = ax.imshow(
    china_share.values,
    aspect="auto",
    cmap=cmap,
    vmin=0,
    vmax=100
)

# X-axis (rotated sector labels)
ax.set_xticks(np.arange(china_share.shape[1]))
ax.set_xticklabels(
    [wrap_label(c, WRAP_WIDTH) for c in china_share.columns],
    rotation=45,
    ha="right",
    fontsize=9
)

# Y-axis (GDP countries)
ax.set_yticks(np.arange(china_share.shape[0]))
ax.set_yticklabels(china_share.index, fontsize=10)

# Gridlines
for j in range(china_share.shape[1] - 1):
    ax.axvline(j + 0.5, color="grey", lw=0.6, alpha=0.3)
for i in range(china_share.shape[0] - 1):
    ax.axhline(i + 0.5, color="grey", lw=0.6, alpha=0.3)

# ==================== ANNOTATE (CHINA-DOMINANT ONLY) ====================
if ANNOTATE:
    for i in range(china_share.shape[0]):
        for j in range(china_share.shape[1]):
            val = china_share.iat[i, j]
            if pd.isna(val) or val < CHINA_DOMINANCE_THRESHOLD:
                continue

            ax.text(
                j, i,
                f"{val:.{ANNOTATION_DECIMALS}f}",
                ha="center",
                va="center",
                fontsize=8,
                color=TEXT_COLOR_DARK if val >= 50 else TEXT_COLOR_LIGHT,
                zorder=5
            )

ax.set_title(
    "China’s sector-specific upstream share in national solar output exposure\n"
    "Selected countries · Values shown only when China’s share ≥ 30%",
    fontsize=14,
    pad=14
)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("China share of upstream exposure (%)")

plt.subplots_adjust(bottom=0.35)
plt.savefig(OUT_PNG, bbox_inches="tight", dpi=300)
plt.close()

print(f"[Saved] {OUT_PNG}")

[Loading master CSV]
[Saved] china_sector_dependence\china_sector_dependence_solar_topGDP.png
